# J-Space Experiment 01 — Hallucination Detection via Workspace State

This notebook implements the first J-space experiment from my personal research programme:

- **H.1 — Workspace Cleanness vs. Answer Correctness**
- **H.3 — Confident Hallucinations Stress Test**

Research question:

> Do J-space activation patterns at answer time contain a signal for factual-answer correctness that is complementary to output confidence?


---


## 0. Configuration



In [ ]:
# ── Experiment identity ───────────────────────────────────────────────────
EXPERIMENT_ID = "01_hallucination_detection"
EXPERIMENT_NAME = "Hallucination Detection via J-Space State"
NOTEBOOK_VERSION = "2.2.1"               # Bumped: registry refactor.
ARTIFACT_SCHEMA_VERSION = "jspace_artifacts_v1"

# ── Dataset selection (dataset_id is the only knob) ──────────
# One of the 7 dataset_ids registered in jspace_adapters.ADAPTERS:
#   triviaqa_rc_nocontext   (the original baseline; alias-rich closed-book)
#   popqa                   (cleanest long-tail factual; popularity field)
#   nq_open                 (real Google queries; alias-poor)
#   truthfulqa_gen          (adversarial misconception; needs prefix-match)
#   hotpotqa_distractor     (multi-hop closed-book; type/level metadata)
#   gsm8k                   (numeric; max_new_tokens=200)
#   commonsenseqa           (MC; letter-output prompt)
DATASET_ID = "triviaqa_rc_nocontext"
DATASET_ROLE = "dev"                  # dev | validation | heldout_test | transfer_test

# ── Model / lens ──────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3-4B"  # Change to Qwen/Qwen3-8B, Qwen/Qwen3-14B, etc. if needed.
MODEL_DTYPE = "auto"             # "auto" -> bf16 if supported, else fp16.
PREFITTED_LENS_REPO = "neuronpedia/jacobian-lens"
ALLOW_FALLBACK_FIT = False       # Keep false for this experiment.

# ── Runtime / run mode ─────────────────────────────────────────────────────
RUN_MODE = "pilot"               # "pilot" or "full"
N_EXAMPLES_PILOT = 50
N_EXAMPLES_FULL = 2000
RANDOM_SEED = 42
SAVE_EVERY = 10
MAX_SEQ_LEN = 512

# ── J-space feature extraction ────────────────────────────────────────────
WORKSPACE_LAYER_FRAC_START = 0.35
WORKSPACE_LAYER_FRAC_END = 0.99
MID_WORKSPACE_FRAC_START = 0.45
MID_WORKSPACE_FRAC_END = 0.75
LATE_WORKSPACE_FRAC_START = 0.75
LATE_WORKSPACE_FRAC_END = 0.99
# Narrower late_late band, focused on the strongest signal
# region (per the PopQA layer-level analysis: layers 30-34 on 36-layer
# Qwen3 carry the largest wrong-vs-correct entropy gap).
LATE_LATE_WORKSPACE_FRAC_START = 0.85
LATE_LATE_WORKSPACE_FRAC_END = 0.99
# Number of trailing layers used for the commitment-dynamic features
# (e.g. entropy slope over the last N layers, max-minus-mean of top-1
# dominance). N=3 is the default that captures the flip from mid- to
# late-workspace in the layer-level analysis.
COMMITMENT_LOOKBACK_LAYERS = 3
# The single layer used for the layer_l33_single ablation.
# Should be one of the layers in workspace_layers_late_late (L30-L34 for
# Qwen3-4B). L33 is the layer with the strongest wrong-vs-correct signal
# per previous analysis (Cohen's d 1.66, AUC 0.906).
# Set to None to disable the layer_l33_* ablations entirely.
LAYER_L33_TARGET_LAYER = 33
LAYER_L34_TARGET_LAYER = 34
LAYER_STRIDE = 1
FEATURE_TOP_K = 50
TOP_TOKENS_TO_SAVE = 5
COMPUTE_FULL_WORKSPACE_ENTROPY = False
COMPUTE_FULL_WORKSPACE_PROBS = False
PIN_LENS_TO_GPU = "auto"

# ── Labeling / adjudication ───────────────────────────────────────────────
USE_LLM_LABEL_JUDGE = True
LABEL_JUDGE_MAX_NEW_TOKENS = 192
LABEL_JUDGE_CONFIDENCE_THRESHOLD = 0.85
LABEL_JUDGE_F1_THRESHOLD = 0.50
LABEL_JUDGE_ACCEPT_CONFIDENCES = ["high", "medium"]
LABEL_JUDGE_REJECT_AMBIGUOUS_FROM_PRIMARY = True
EXCLUDE_TEMPORAL_UNSTABLE_FROM_PRIMARY = True
TEMPORAL_UNSTABLE_REGEX = r"\b(current|currently|today|recently|this year|latest)\b"
FUZZY_AUTO_ACCEPT = False
FUZZY_MATCH_THRESHOLD = 0.92
# When True, skip the LLM judge on high-confidence + low-F1
# (deterministically-wrong) cases. The judge is only called for
# genuinely uncertain cases (fuzzy match, alternative-separator, etc.).
LLM_JUDGE_SKIP_HIGH_CONF_WRONG = False

# ── Analysis ───────────────────────────────────────────────────────────────
CV_FOLDS = 5
THRESHOLD_MODE = "exploratory_run_median"  # exploratory_run_median | frozen_absolute
PROTOCOL_FILE = "/content/drive/MyDrive/J-Space_Artifacts/workspace_backup/jspace_workbench/protocols/H3_overconfident_workspace_stratification_v1.DRAFT.json"
HIGH_CONFIDENCE_QUANTILE = 0.50
NOISY_WORKSPACE_QUANTILE = 0.50
MIN_CLASS_COUNT_FOR_CV = 5
SELECTED_EXAMPLES_PER_BUCKET = 10
BOOTSTRAP_N = 1000
PERMUTATION_N = 500

# ── Artefact storage ───────────────────────────────────────────────────────
USE_GOOGLE_DRIVE = True
MOUNT_GOOGLE_DRIVE = True
DRIVE_ARTIFACT_ROOT = "/content/drive/MyDrive/J-Space_Artifacts"
LOCAL_ARTIFACT_ROOT = "J-Space_Artifacts"
SOURCE_NOTEBOOK_PATH = None
SAVE_NOTEBOOK_COPY = True

# ── Retention flags ────────────────────────────────────────────────────────
STORE_FULL_PROMPTS = True
STORE_FULL_OUTPUTS = True
STORE_SELECTED_EXAMPLES_TEXT = True

# ── Optional display controls ──────────────────────────────────────────────
VERBOSE_EXAMPLES = True
USE_DISPLAY_FILTERS = False
FILTER_SPECIAL_TOKENS = True
FILTER_WHITESPACE_ONLY = True
FILTER_PUNCTUATION_ONLY = True
FILTER_NONPRINTABLE = True

N_EXAMPLES = N_EXAMPLES_PILOT if RUN_MODE == "pilot" else N_EXAMPLES_FULL

print(f"Experiment: {EXPERIMENT_ID} — {EXPERIMENT_NAME}")
print(f"Notebook version: {NOTEBOOK_VERSION} (registry refactor)")
print(f"Run mode: {RUN_MODE}, N_EXAMPLES={N_EXAMPLES}")
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_ID} (role={DATASET_ROLE})")
print(f"Artefact root target: {DRIVE_ARTIFACT_ROOT if USE_GOOGLE_DRIVE else LOCAL_ARTIFACT_ROOT}")


## 1. GPU Verification

Checks that a CUDA GPU is available and reports VRAM and bf16 support.


In [ ]:
import os
import re
import sys
import time
import math
import random
import string
import hashlib
import json
import uuid
import shutil
from pathlib import Path
from datetime import datetime, timezone

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_free = torch.cuda.mem_get_info(0)[0] / 1e9
cuda_version = torch.version.cuda
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU:             {gpu_name}")
print(f"VRAM total:      {vram_total:.1f} GB")
print(f"VRAM free:       {vram_free:.1f} GB")
print(f"CUDA:            {cuda_version}")
print(f"PyTorch:         {torch.__version__}")
print(f"Python:          {sys.version}")
print(f"bf16 supported:  {bf16_supported}")

match = re.search(r"(\d+(?:\.\d+)?)B", MODEL_NAME, flags=re.IGNORECASE)
params = float(match.group(1)) if match else 4.0
est_weight_gb = params * 2
print(f"\nEstimated parameter memory at fp16/bf16: ~{est_weight_gb:.1f} GB")
if est_weight_gb > vram_total * 0.85:
    print("⚠️  Model weights alone may be too large for this GPU.")
elif est_weight_gb > vram_total * 0.65:
    print("⚠️  Weights should fit, but headroom is limited. Keep prompts short.")
else:
    print("✅ Model weights should fit comfortably.")


## 1.5 Initialize Artefact Run

Creates a unique artefact folder and writes initial metadata/status files.


In [ ]:
def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def safe_slug(text: str, max_len: int = 80) -> str:
    text = str(text).lower().replace("/", "-").replace(".", "p")
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text[:max_len].strip("-") or "unknown"


def sha256_file(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_write_text(path: Path, text: str, encoding="utf-8"):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".tmp.{os.getpid()}")
    tmp.write_text(text, encoding=encoding)
    os.replace(tmp, path)


class ArtifactRun:
    """Artefact manager used by this and future J-space experiment notebooks."""

    def __init__(self, *, root: Path, experiment_id: str, experiment_name: str, run_id: str, config: dict):
        self.root = Path(root)
        self.experiment_id = experiment_id
        self.experiment_name = experiment_name
        self.run_id = run_id
        self.run_path = self.root / "experiments" / experiment_id / "runs" / run_id
        self.created_at_utc = utc_now_iso()
        self.artifacts = {}
        self.completed_steps = []
        self.append_only_paths = set()

        for subdir in [
            "notebook", "logs", "prompts", "datasets", "generations", "scans", "metrics",
            "examples", "plots", "arrays", "interventions", "training",
        ]:
            (self.run_path / subdir).mkdir(parents=True, exist_ok=True)
        (self.root / "_index").mkdir(parents=True, exist_ok=True)
        (self.root / "_shared" / "schema_versions").mkdir(parents=True, exist_ok=True)

        self.manifest = {
            "schema_version": ARTIFACT_SCHEMA_VERSION,
            "run_id": run_id,
            "experiment_id": experiment_id,
            "experiment_name": experiment_name,
            "notebook_version": NOTEBOOK_VERSION,
            "created_at_utc": self.created_at_utc,
            "timezone": "Europe/Berlin",
            "status": "running",
            "run_path": str(self.run_path),
            "artifact_root": str(self.root),
        }
        self.status = {
            "schema_version": ARTIFACT_SCHEMA_VERSION,
            "run_id": run_id,
            "status": "running",
            "started_at_utc": self.created_at_utc,
            "last_updated_utc": self.created_at_utc,
            "completed_steps": [],
        }
        self.write_json("manifest.json", self.manifest, artifact_type="metadata", description="Run manifest.")
        self.write_json("status.json", self.status, artifact_type="metadata", description="Incremental run status.")
        self.write_json("config.json", config, artifact_type="config", description="Notebook configuration at run start.")
        self.log_event("run_created", {"run_id": run_id, "run_path": str(self.run_path)})

    def full_path(self, rel_path):
        return self.run_path / rel_path

    def register_artifact(self, rel_path, *, artifact_type, fmt=None, description="", extra=None, compute_hash=True):
        rel = str(rel_path).replace("\\", "/")
        path = self.full_path(rel)
        stat = path.stat() if path.exists() and path.is_file() else None
        item = {
            "path": rel,
            "type": artifact_type,
            "format": fmt or Path(rel).suffix.lstrip(".") or "unknown",
            "description": description,
            "created_or_updated_at_utc": utc_now_iso(),
            "bytes": stat.st_size if stat else None,
            "sha256": sha256_file(path) if (stat and compute_hash) else None,
        }
        if extra:
            item.update(extra)
        self.artifacts[rel] = item
        self._write_artifact_indexes(compute_missing_hashes=False)
        return item

    def _write_artifact_indexes(self, compute_missing_hashes=False):
        if compute_missing_hashes:
            for rel, item in list(self.artifacts.items()):
                path = self.full_path(rel)
                if path.exists() and path.is_file():
                    stat = path.stat()
                    item["bytes"] = stat.st_size
                    item["sha256"] = sha256_file(path)
                    self.artifacts[rel] = item
        records = list(self.artifacts.values())
        atomic_write_text(self.full_path("artifacts.json"), json.dumps(records, indent=2, ensure_ascii=False))
        atomic_write_text(self.full_path("artifacts.jsonl"), "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in records))

    def write_text(self, rel_path, text, *, artifact_type="text", description=""):
        path = self.full_path(rel_path)
        atomic_write_text(path, text)
        return self.register_artifact(rel_path, artifact_type=artifact_type, description=description)

    def write_json(self, rel_path, obj, *, artifact_type="json", description=""):
        path = self.full_path(rel_path)
        atomic_write_text(path, json.dumps(obj, indent=2, ensure_ascii=False, default=str))
        return self.register_artifact(rel_path, artifact_type=artifact_type, fmt="json", description=description)

    def append_jsonl(self, rel_path, record, *, artifact_type="jsonl", description=""):
        path = self.full_path(rel_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
            f.flush()
        if rel_path not in self.artifacts:
            self.register_artifact(rel_path, artifact_type=artifact_type, fmt="jsonl", description=description, compute_hash=False)
        self.append_only_paths.add(rel_path)

    def write_dataframe(self, rel_base, df, *, description=""):
        csv_rel = rel_base + ".csv"
        jsonl_rel = rel_base + ".jsonl"
        csv_path = self.full_path(csv_rel)
        jsonl_path = self.full_path(jsonl_rel)
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv_path, index=False)
        records = df.to_dict(orient="records")
        atomic_write_text(jsonl_path, "".join(json.dumps(r, ensure_ascii=False, default=str) + "\n" for r in records))
        self.register_artifact(csv_rel, artifact_type="table", fmt="csv", description=description, extra={"rows": len(df)})
        self.register_artifact(jsonl_rel, artifact_type="table", fmt="jsonl", description=description, extra={"rows": len(df)})

    def log_event(self, event_type, payload=None):
        self.append_jsonl("logs/events.jsonl", {"timestamp_utc": utc_now_iso(), "event_type": event_type, "payload": payload or {}}, artifact_type="log", description="Run event log.")

    def update_manifest_section(self, key, value):
        self.manifest[key] = value
        self.manifest["last_updated_utc"] = utc_now_iso()
        self.write_json("manifest.json", self.manifest, artifact_type="metadata", description="Run manifest.")

    def update_status(self, *, step=None, status=None, extra=None):
        if step and step not in self.completed_steps:
            self.completed_steps.append(step)
        if status:
            self.status["status"] = status
            self.manifest["status"] = status
        self.status["completed_steps"] = list(self.completed_steps)
        self.status["last_updated_utc"] = utc_now_iso()
        if extra:
            self.status.update(extra)
        self.write_json("status.json", self.status, artifact_type="metadata", description="Incremental run status.")
        self.write_json("manifest.json", self.manifest, artifact_type="metadata", description="Run manifest.")

    def copy_notebook_if_configured(self):
        if not SAVE_NOTEBOOK_COPY:
            return
        note = []
        if SOURCE_NOTEBOOK_PATH:
            src = Path(SOURCE_NOTEBOOK_PATH)
            if src.exists():
                dst = self.full_path("notebook/source_notebook.ipynb")
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
                self.register_artifact("notebook/source_notebook.ipynb", artifact_type="notebook", fmt="ipynb", description="Copied source notebook.")
                note.append(f"Copied source notebook from {src}")
            else:
                note.append(f"SOURCE_NOTEBOOK_PATH was set but not found: {src}")
        else:
            note.append("SOURCE_NOTEBOOK_PATH was not set. Colab does not reliably expose the currently executed notebook path programmatically.")
        self.write_text("notebook/notebook_capture_note.md", "\n".join(f"- {x}" for x in note) + "\n", artifact_type="notebook_note", description="Notebook capture note.")
        self.write_json("notebook/notebook_metadata.json", {"source_notebook_path": SOURCE_NOTEBOOK_PATH, "note": note}, artifact_type="notebook_metadata", description="Notebook capture metadata.")

    def finalize(self, *, overall_result, final_summary):
        completed_at = utc_now_iso()
        self.status.update({"status": "completed", "completed_at_utc": completed_at, "last_updated_utc": completed_at, "overall_result": overall_result, "final_summary": final_summary})
        self.manifest.update({"status": "completed", "completed_at_utc": completed_at, "overall_result": overall_result, "final_summary": final_summary})
        self.copy_notebook_if_configured()
        self.write_json("status.json", self.status, artifact_type="metadata", description="Final run status.")
        self.write_json("manifest.json", self.manifest, artifact_type="metadata", description="Final run manifest.")
        # Log finalization before final artifact hash pass so logs/events.jsonl hash is current.
        self.log_event("run_finalized", {"overall_result": overall_result})
        self._write_artifact_indexes(compute_missing_hashes=True)
        self.update_global_index(overall_result=overall_result)

    def update_global_index(self, *, overall_result):
        index_record = {
            "run_id": self.run_id,
            "experiment_id": self.experiment_id,
            "experiment_name": self.experiment_name,
            "model_name": self.manifest.get("model", {}).get("name"),
            "lens_filename": self.manifest.get("lens", {}).get("filename"),
            "gpu": self.manifest.get("runtime", {}).get("gpu"),
            "status": "completed",
            "overall_result": overall_result,
            "created_at_utc": self.created_at_utc,
            "completed_at_utc": self.status.get("completed_at_utc"),
            "run_path": str(self.run_path),
        }
        runs_index = self.root / "_index" / "runs_index.jsonl"
        runs_index.parent.mkdir(parents=True, exist_ok=True)
        with runs_index.open("a", encoding="utf-8") as f:
            f.write(json.dumps(index_record, ensure_ascii=False) + "\n")
        latest_path = self.root / "_index" / "latest_by_experiment.json"
        latest = json.loads(latest_path.read_text(encoding="utf-8")) if latest_path.exists() else {}
        latest[self.experiment_id] = {"latest_completed_run_id": self.run_id, "latest_completed_run_path": str(self.run_path), "completed_at_utc": self.status.get("completed_at_utc"), "overall_result": overall_result}
        atomic_write_text(latest_path, json.dumps(latest, indent=2, ensure_ascii=False))
        try:
            import pandas as _pd
            records = [json.loads(line) for line in runs_index.read_text(encoding="utf-8").splitlines() if line.strip()]
            _pd.DataFrame(records).to_csv(self.root / "_index" / "runs_index.csv", index=False)
        except Exception:
            pass


# Choose artefact root.
if USE_GOOGLE_DRIVE:
    try:
        if MOUNT_GOOGLE_DRIVE:
            from google.colab import drive
            if not Path("/content/drive/MyDrive").exists():
                drive.mount("/content/drive")
        artifact_root = Path(DRIVE_ARTIFACT_ROOT)
        artifact_storage_note = "google_drive"
    except Exception as e:
        artifact_root = Path(LOCAL_ARTIFACT_ROOT)
        artifact_storage_note = f"local_fallback_after_drive_error: {type(e).__name__}: {e}"
else:
    artifact_root = Path(LOCAL_ARTIFACT_ROOT)
    artifact_storage_note = "local_configured"

run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id = f"{run_timestamp}_{safe_slug(MODEL_NAME)}_{safe_slug(gpu_name, 30)}_{uuid.uuid4().hex[:8]}"

def _jsonable_config_value(v):
    if isinstance(v, (str, int, float, bool)) or v is None:
        return v
    if isinstance(v, (list, tuple)) and all(isinstance(x, (str, int, float, bool)) or x is None for x in v):
        return list(v)
    if isinstance(v, dict) and all(isinstance(k, str) and (isinstance(x, (str, int, float, bool)) or x is None) for k, x in v.items()):
        return dict(v)
    return None

initial_config = {}
for k, v in list(globals().items()):
    if k.isupper():
        jv = _jsonable_config_value(v)
        if jv is not None or v is None:
            initial_config[k] = jv
initial_config.update({"artifact_storage_note": artifact_storage_note})

artifact_run = ArtifactRun(root=artifact_root, experiment_id=EXPERIMENT_ID, experiment_name=EXPERIMENT_NAME, run_id=run_id, config=initial_config)
artifact_run.update_manifest_section("runtime_initial", {"gpu": gpu_name, "vram_total_gb": vram_total, "vram_free_gb_at_start": vram_free, "cuda": cuda_version, "torch": torch.__version__, "python": sys.version, "bf16_supported": bf16_supported})
artifact_run.update_status(step="artifact_run_initialized")

print("✅ Artefact run initialized.")
print(f"Run ID: {artifact_run.run_id}")
print(f"Run path: {artifact_run.run_path}")
print(f"Storage note: {artifact_storage_note}")


## 2. Install Dependencies

Installs `jlens`, HuggingFace tooling, and analysis packages.


In [ ]:
!pip install -q git+https://github.com/anthropics/jacobian-lens.git
!pip install -q transformers datasets accelerate huggingface_hub pandas scikit-learn scipy matplotlib

from importlib.metadata import PackageNotFoundError, version

import datasets
import jlens
import numpy as np
import pandas as pd
import transformers

try:
    JLENS_VERSION = version("jlens")
except PackageNotFoundError:
    JLENS_VERSION = "unknown"

print(f"jlens version: {JLENS_VERSION}")
print(f"transformers:  {transformers.__version__}")
print(f"datasets:      {datasets.__version__}")
print(f"pandas:        {pd.__version__}")

artifact_run.update_manifest_section("packages", {"jlens": JLENS_VERSION, "transformers": transformers.__version__, "datasets": datasets.__version__, "pandas": pd.__version__})
artifact_run.log_event("dependencies_installed", {"jlens": JLENS_VERSION, "transformers": transformers.__version__, "datasets": datasets.__version__, "pandas": pd.__version__})
artifact_run.update_status(step="dependencies_installed")
print("✅ Dependencies ready.")


## 3. Load Model and J-Lens

Loads the selected model and matching pre-fitted Neuronpedia lens.


In [ ]:
import jlens
from huggingface_hub import list_repo_files

if str(MODEL_DTYPE).lower() == "auto":
    resolved_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif str(MODEL_DTYPE).lower() in {"bf16", "bfloat16"}:
    resolved_dtype = torch.bfloat16
elif str(MODEL_DTYPE).lower() in {"fp16", "float16", "half"}:
    resolved_dtype = torch.float16
else:
    raise ValueError(f"Unsupported MODEL_DTYPE={MODEL_DTYPE!r}")

print(f"Loading {MODEL_NAME} with dtype={resolved_dtype}...")
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=resolved_dtype,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = jlens.from_hf(hf_model, tokenizer)

n_layers = model.n_layers
d_model = model.d_model
vocab_size = len(tokenizer)
first_device = next(hf_model.parameters()).device
vram_used = torch.cuda.memory_allocated(0) / 1e9
print(f"Model loaded: layers={n_layers}, d_model={d_model}, vocab={vocab_size}, VRAM={vram_used:.1f} GB")

KNOWN_NEURONPEDIA_LENS_FILES = {
    "qwen/qwen3-1.7b": "qwen3-1.7b/jlens/Salesforce-wikitext/Qwen3-1.7B_jacobian_lens.pt",
    "qwen/qwen3-4b": "qwen3-4b/jlens/Salesforce-wikitext/Qwen3-4B_jacobian_lens.pt",
    "qwen/qwen3-8b": "qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt",
    "qwen/qwen3-14b": "qwen3-14b/jlens/Salesforce-wikitext/Qwen3-14B_jacobian_lens.pt",
    "qwen/qwen3-32b": "qwen3-32b/jlens/Salesforce-wikitext/Qwen3-32B_jacobian_lens.pt",
    "qwen/qwen3.5-4b": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens.pt",
    "qwen/qwen3.5-27b": "qwen3.5-27b/jlens/Salesforce-wikitext/Qwen3.5-27B_jacobian_lens.pt",
    "qwen/qwen3.6-27b": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}


def find_prefitted_lens_filename(repo_id, model_name):
    key = model_name.lower()
    if key in KNOWN_NEURONPEDIA_LENS_FILES:
        return KNOWN_NEURONPEDIA_LENS_FILES[key]
    short = model_name.split("/")[-1].lower().replace("_", "-")
    files = list_repo_files(repo_id, repo_type="model")
    candidates = []
    for fname in files:
        fl = fname.lower()
        if not fl.endswith(".pt"):
            continue
        if "jacobian_lens" not in fl and not fl.endswith("lens.pt"):
            continue
        if short in fl:
            score = 10 + int("n1000" in fl) + int("salesforce-wikitext" in fl)
            candidates.append((score, fname))
    candidates.sort(reverse=True)
    return candidates[0][1] if candidates else None

lens_filename = find_prefitted_lens_filename(PREFITTED_LENS_REPO, MODEL_NAME)
if lens_filename is None:
    if not ALLOW_FALLBACK_FIT:
        raise RuntimeError("No pre-fitted lens found and fallback fitting is disabled.")
    raise NotImplementedError("Fallback fitting is intentionally not implemented in this experiment notebook.")

print(f"Loading lens: {lens_filename}")
lens = jlens.JacobianLens.from_pretrained(PREFITTED_LENS_REPO, filename=lens_filename)
if lens.d_model != d_model:
    raise RuntimeError(f"Lens d_model={lens.d_model} does not match model d_model={d_model}")
print(f"Lens loaded: source_layers={len(lens.source_layers)}, layer_range={lens.source_layers[0]}–{lens.source_layers[-1]}, d_model={lens.d_model}")

# Optional performance optimization: pin lens matrices to GPU if enough headroom.
def maybe_pin_lens_to_gpu(lens, setting="auto"):
    if setting is False or str(setting).lower() == "false":
        return "not_pinned_config_false"
    if not torch.cuda.is_available():
        return "not_pinned_no_cuda"
    lens_gb = sum(J.numel() * J.element_size() for J in lens.jacobians.values()) / 1e9
    free_gb = torch.cuda.mem_get_info(0)[0] / 1e9
    should_pin = bool(setting is True or str(setting).lower() == "true" or (str(setting).lower() == "auto" and free_gb > lens_gb + 4.0))
    if not should_pin:
        return f"not_pinned_free_gb={free_gb:.1f}_lens_gb={lens_gb:.1f}"
    for layer in list(lens.jacobians.keys()):
        lens.jacobians[layer] = lens.jacobians[layer].to(first_device)
    return f"pinned_to_{first_device}_lens_gb={lens_gb:.1f}"

lens_pin_status = maybe_pin_lens_to_gpu(lens, PIN_LENS_TO_GPU)
print(f"Lens pin status: {lens_pin_status}")

model_metadata = {"name": MODEL_NAME, "dtype": str(resolved_dtype), "n_layers": n_layers, "d_model": d_model, "vocab_size": vocab_size, "first_device": str(first_device), "vram_used_after_load_gb": vram_used}
lens_metadata = {"source": f"Pre-fitted from {PREFITTED_LENS_REPO}", "repo": PREFITTED_LENS_REPO, "filename": lens_filename, "type": type(lens).__name__, "source_layers_count": len(lens.source_layers), "source_layer_first": lens.source_layers[0], "source_layer_last": lens.source_layers[-1], "source_layers": list(lens.source_layers), "d_model": lens.d_model, "n_prompts": getattr(lens, "n_prompts", None), "pin_status": lens_pin_status}
runtime_metadata = {"gpu": gpu_name, "vram_total_gb": vram_total, "cuda": cuda_version, "torch": torch.__version__, "python": sys.version, "bf16_supported": bf16_supported, "transformers": transformers.__version__, "jlens": JLENS_VERSION}
artifact_run.update_manifest_section("model", model_metadata)
artifact_run.update_manifest_section("lens", lens_metadata)
artifact_run.update_manifest_section("runtime", runtime_metadata)
artifact_run.write_json("model_lens.json", {"model": model_metadata, "lens": lens_metadata}, artifact_type="metadata", description="Model and lens metadata.")
artifact_run.write_json("environment.json", runtime_metadata, artifact_type="metadata", description="Runtime and package metadata.")
artifact_run.log_event("model_and_lens_loaded", {"model": model_metadata, "lens": lens_metadata})
artifact_run.update_status(step="model_and_lens_loaded")


## 4. Shared Helpers

Defines prompt formatting, answer normalization, correctness labeling, J-space feature extraction, and report utilities.


In [ ]:
import sys

# ── Adapter registry imports ──────────────────────────────
# The registry is on Google Drive at the same path as the artefact root.
# Adjust this path if running locally.
_REGISTRY_PARENT = "/content/drive/MyDrive/J-Space_Artifacts/_shared/utils"
if _REGISTRY_PARENT not in sys.path:
    sys.path.insert(0, _REGISTRY_PARENT)
from jspace_adapters import get_adapter, list_adapter_ids  # noqa: E402
from jspace_adapters.base import JspaceExample, safe_slug   # noqa: E402
from jspace_adapters.labelling import (                     # noqa: E402
    normalize_answer, token_f1,
    deterministic_label, deterministic_label_mc,
    deterministic_label_with_prefix,
)
from jspace_adapters.prompt_templates import (               # noqa: E402
    parse_short_answer, parse_mc_letter, parse_numeric,
)
from jspace_adapters.selection import select_deduplicated   # noqa: E402

import unicodedata
from collections import Counter
from difflib import SequenceMatcher

try:
    from IPython.display import display
except Exception:
    display = None

validation_results = []
prompt_records = {}
generation_records = []
workspace_aggregate_records = []
workspace_layer_records = []
per_example_records = []
selected_example_records = []
label_judge_records = []

TRIVIAQA_PUNCT_EXCLUDE = set(
    string.punctuation
    + "".join(["\u2018", "\u2019", "\u00b4", "`", "\u201c", "\u201d"])
)


def decode_token(token_id):
    return tokenizer.decode([int(token_id)], clean_up_tokenization_spaces=False)


def model_token_ids(text, max_seq_len=MAX_SEQ_LEN):
    return model.encode(text, max_length=max_seq_len)[0].detach().cpu().tolist()


def record_prompt(prompt_id, condition_id, prompt_text, *, prompt_type="raw_completion", spans=None, extra=None):
    toks = model_token_ids(prompt_text)
    rec = {
        "prompt_id": prompt_id,
        "condition_id": condition_id,
        "prompt_type": prompt_type,
        "token_count": len(toks),
        "tokens": [decode_token(t) for t in toks],
        "spans": spans or {},
        "extra": extra or {},
    }
    if STORE_FULL_PROMPTS:
        rec["prompt_text"] = prompt_text
    else:
        rec["prompt_sha256"] = hashlib.sha256(prompt_text.encode("utf-8")).hexdigest()
    prompt_records[prompt_id] = rec
    artifact_run.append_jsonl("prompts/prompts.jsonl", rec, artifact_type="prompt", description="Prompt records.")
    return rec


def dataframe_to_markdown(df):
    if df is None or len(df) == 0:
        return "_(empty table)_\n"
    cols = list(df.columns)
    def cell(x):
        return ("" if x is None else str(x)).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(cell(row[c]) for c in cols) + " |")
    return "\n".join(lines) + "\n"


def write_text_report(rel_path, title, sections):
    lines = [f"# {title}\n"]
    for h, body in sections:
        lines.append(f"## {h}\n")
        lines.append(str(body).rstrip() + "\n")
    artifact_run.write_text(rel_path, "\n".join(lines), artifact_type="report", description=f"Report: {title}")


# ── Adapter-based prompt building ──────────────────────────────────────────
# The old build_prompt() took only a question string and used a hardcoded
# PROMPT_MODE. The new version delegates to the adapter, which knows the
# dataset's prompt template id, max_new_tokens, and (for MC) the choices.

def build_prompt(example, *, tokenizer=tokenizer):
    """Build the prompt text for one JspaceExample using its adapter.

    For backward compatibility, the argument can also be a plain question
    string, in which case we wrap it in a minimal JspaceExample and use the
    default short_answer_v1 template.
    """
    if isinstance(example, str):
        ex = JspaceExample(question=example)
        return get_adapter("triviaqa_rc_nocontext").build_prompt(ex)
    if isinstance(example, dict):
        ex = JspaceExample(
            question=example.get("question", ""),
            choices=example.get("choices"),
            correct_choice_label=example.get("correct_choice_label"),
            correct_choice_text=example.get("correct_choice_text"),
        )
        return get_adapter("triviaqa_rc_nocontext").build_prompt(ex)
    return get_adapter(DATASET_ID).build_prompt(example)


# ── LLM judge and orchestration ───────────────────────────────────────────

def detect_temporal_unstable(question):
    return bool(re.search(TEMPORAL_UNSTABLE_REGEX, question or "", flags=re.I))


def should_run_label_judge(det, question, output_confidence):
    if not USE_LLM_LABEL_JUDGE:
        return False
    if det["deterministic_correct"]:
        return False
    if LLM_JUDGE_SKIP_HIGH_CONF_WRONG:
        f1 = float(det.get("token_f1_max", 0.0) or 0.0)
        if (f1 < LABEL_JUDGE_F1_THRESHOLD
                and output_confidence is not None
                and output_confidence >= LABEL_JUDGE_CONFIDENCE_THRESHOLD):
            return False
    q = (question or "").lower()
    if det.get("fuzzy_suggested"):
        return True
    if det["token_f1_max"] >= LABEL_JUDGE_F1_THRESHOLD:
        return True
    if output_confidence is not None and output_confidence >= LABEL_JUDGE_CONFIDENCE_THRESHOLD:
        return True
    if det.get("alias_has_alternative_sep"):
        return True
    if detect_temporal_unstable(question):
        return True
    if re.search(r"\b(either|one of|any of|famously|also known|better known|which city|which country)\b", q):
        return True
    return False


def build_judge_prompt(question, prediction, aliases):
    aliases_for_prompt = aliases[:30]
    temporal_note = (
        "\nThis question appears time-sensitive/current; if the model answer is correct in the present day but conflicts with stale aliases, use verdict='ambiguous'.\n"
        if detect_temporal_unstable(question) else ""
    )
    return (
        "You are a strict but fair evaluator for a trivia question-answering benchmark.\n"
        "Decide whether the model answer should be counted as correct.\n\n"
        "Rules:\n"
        "- Mark correct if the model answer is semantically equivalent to an accepted answer, or is a valid shorter form of a compound/list answer.\n"
        "- Mark ambiguous if the accepted aliases appear incomplete, historically disputed, stale/time-sensitive, or the model answer is partially correct but not clearly equivalent.\n"
        "- Mark incorrect if the answer is clearly a different entity/fact under both the aliases and general knowledge.\n"
        "- If the question asks for 'current', 'today', 'this year', or similar time-dependent facts and the aliases may be stale, prefer ambiguous over incorrect.\n"
        "- Be concise. Return JSON only.\n"
        f"{temporal_note}\n"
        f"Question: {question}\n"
        f"Accepted answers / aliases: {json.dumps(aliases_for_prompt, ensure_ascii=False)}\n"
        f"Model answer: {prediction}\n\n"
        'Return exactly this JSON schema:\n'
        '{"verdict":"correct|incorrect|ambiguous","confidence":"high|medium|low","reason":"..."}'
    )


def generate_judge_verdict(question, prediction, aliases):
    user_content = build_judge_prompt(question, prediction, aliases)
    messages = [{"role": "user", "content": user_content}]
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except Exception:
        prompt = user_content + "\nJSON:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(first_device)
    gen_kwargs = dict(
        max_new_tokens=LABEL_JUDGE_MAX_NEW_TOKENS, do_sample=False,
        return_dict_in_generate=True, output_scores=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    if FORBID_THINKING_TOKENS:
        suppress_ids = []
        for tok in ["<think>", "</think>"]:
            ids = tokenizer.encode(tok, add_special_tokens=False)
            if len(ids) == 1:
                suppress_ids.append(int(ids[0]))
        if suppress_ids:
            gen_kwargs["suppress_tokens"] = sorted(set(suppress_ids))
    with torch.no_grad():
        out = hf_model.generate(**inputs, **gen_kwargs)
    new_ids = out.sequences[0][inputs.input_ids.shape[1]:].detach().cpu().tolist()
    raw = tokenizer.decode(new_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    clean = strip_thinking_text(raw).strip()
    parsed = None
    json_match = re.search(r"\{.*\}", clean, flags=re.DOTALL)
    if json_match:
        try:
            parsed = json.loads(json_match.group(0))
        except Exception:
            parsed = None
    if not isinstance(parsed, dict):
        low = clean.lower()
        if "ambiguous" in low:
            verdict = "ambiguous"
        elif "incorrect" in low or "wrong" in low:
            verdict = "incorrect"
        elif "correct" in low:
            verdict = "correct"
        else:
            verdict = "ambiguous"
        parsed = {"verdict": verdict, "confidence": "low", "reason": clean[:500]}
    verdict = str(parsed.get("verdict", "ambiguous")).lower().strip()
    if verdict not in {"correct", "incorrect", "ambiguous"}:
        verdict = "ambiguous"
    confidence = str(parsed.get("confidence", "low")).lower().strip()
    if confidence not in {"high", "medium", "low"}:
        confidence = "low"
    return {"verdict": verdict, "confidence": confidence, "reason": str(parsed.get("reason", "")), "raw_judge_output": raw, "clean_judge_output": clean}


def label_correctness(
    prediction, aliases_or_example, *, question=None, output_confidence=None, example_id=None,
    det=None, record_judge=True,
):
    """Orchestrate deterministic + LLM-judge + temporal-unstable exclusion.

    `aliases_or_example` can be either a list[str] (backward-compatible)
    or a JspaceExample (new path; the registry uses this). The `det` argument
    is the precomputed deterministic_label result. If None, we compute it
    here from the aliases and prediction (legacy path).
    """
    # Normalize input shape.
    if isinstance(aliases_or_example, JspaceExample):
        ex = aliases_or_example
        aliases = ex.aliases
        if question is None:
            question = ex.question
    else:
        ex = None
        aliases = list(aliases_or_example) if aliases_or_example else []

    if det is None:
        # Legacy path: compute deterministic label from raw aliases.
        det = deterministic_label(prediction, aliases)

    temporal_unstable = detect_temporal_unstable(question or "")
    judge_needed = should_run_label_judge(det, question, output_confidence)
    judge = None
    if judge_needed:
        try:
            judge = generate_judge_verdict(question or "", prediction, aliases)
        except Exception as e:
            judge = {"verdict": "ambiguous", "confidence": "low", "reason": f"judge_error: {type(e).__name__}: {e}", "raw_judge_output": "", "clean_judge_output": ""}
        if record_judge:
            judge_record = {
                "example_id": example_id, "question": question, "prediction": prediction,
                "aliases": aliases[:30], **judge, "temporal_unstable": temporal_unstable,
                "fuzzy_suggested": det.get("fuzzy_suggested"),
            }
            label_judge_records.append(judge_record)
            artifact_run.append_jsonl("metrics/label_judge_records.jsonl", judge_record, artifact_type="label_judge", description="LLM label judge records.")

    if temporal_unstable and EXCLUDE_TEMPORAL_UNSTABLE_FROM_PRIMARY:
        correct_primary = None
        label_source = "temporal_unstable_excluded"
        label_quality = "ambiguous"
        analysis_include_primary = False
        label_exclusion_reason = "temporal_unstable"
    elif det["deterministic_correct"]:
        correct_primary = True
        label_source = det["deterministic_label_source"]
        label_quality = det["deterministic_label_quality"]
        analysis_include_primary = True
        label_exclusion_reason = None
    elif judge is not None and judge["verdict"] == "correct" and judge["confidence"] in set(LABEL_JUDGE_ACCEPT_CONFIDENCES):
        correct_primary = True
        label_source = "llm_judge_correct"
        label_quality = "medium" if judge["confidence"] == "high" else "low"
        analysis_include_primary = True
        label_exclusion_reason = None
    elif judge is not None and judge["verdict"] == "ambiguous" and LABEL_JUDGE_REJECT_AMBIGUOUS_FROM_PRIMARY:
        correct_primary = None
        label_source = "llm_judge_ambiguous_excluded"
        label_quality = "ambiguous"
        analysis_include_primary = False
        label_exclusion_reason = "llm_judge_ambiguous"
    elif judge is not None and judge["verdict"] == "incorrect":
        correct_primary = False
        label_source = "llm_judge_incorrect"
        label_quality = "medium" if judge["confidence"] in {"high", "medium"} else "low"
        analysis_include_primary = True
        label_exclusion_reason = None
    else:
        correct_primary = False
        label_source = det["deterministic_label_source"]
        label_quality = det["deterministic_label_quality"]
        analysis_include_primary = True
        label_exclusion_reason = None

    return {
        **det,
        "exact_match": det["strict_correct"],
        "contains_alias": det["safe_contains_correct"],
        "correct_auto": correct_primary,
        "correct_primary": correct_primary,
        "correct_strict": det["strict_correct"],
        "correct_expanded": bool(
            det["strict_correct"] or det["expanded_exact_correct"]
            or det["safe_contains_correct"] or det["fuzzy_auto_accept"]
        ),
        "correct_deterministic": det["deterministic_correct"],
        "label_source": label_source,
        "label_quality": label_quality,
        "label_confidence": label_quality,
        "match_type": label_source,
        "judge_needed": judge_needed,
        "judge_verdict": judge["verdict"] if judge else None,
        "judge_confidence": judge["confidence"] if judge else None,
        "judge_reason": judge["reason"] if judge else None,
        "temporal_unstable": temporal_unstable,
        "label_exclusion_reason": label_exclusion_reason,
        "analysis_include_primary": analysis_include_primary,
        "analysis_include_high_quality": bool(analysis_include_primary and label_quality in {"high", "medium"}),
    }


# ── Generation, prompt cleaning, J-space feature extraction (unchanged) ──

def is_content_token(token_id):
    text = decode_token(token_id)
    stripped = text.strip().lower()
    if text in getattr(tokenizer, "all_special_tokens", []):
        return False
    if text.startswith("<|") and text.endswith("|>"):
        return False
    if stripped in {"<think>", "</think>", "<think", "</think"}:
        return False
    return bool(text.strip())


def strip_thinking_text(raw_text):
    text = raw_text.strip()
    lower = text.lower()
    if "<think" in lower:
        m = re.search(r"</think>\s*", text, flags=re.IGNORECASE)
        if m:
            text = text[m.end():].strip()
        else:
            return ""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()
    return text


def clean_generated_answer(raw_text):
    text = strip_thinking_text(raw_text)
    text = text.splitlines()[0] if text.splitlines() else text
    text = re.sub(
        r"^(the\s+answer\s+is|answer\s*:|it\s+is|it's)\s+",
        "", text, flags=re.IGNORECASE,
    ).strip()
    return text.strip().strip(" .,:;!?\\\"'`\u201c\u201d\u2018\u2019()[]{}")


FORBID_THINKING_TOKENS = True

def _adapter_max_new_tokens():
    return get_adapter(DATASET_ID).max_new_tokens


def generate_answer_with_confidence(prompt, *, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = _adapter_max_new_tokens()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(first_device)
    input_len = inputs.input_ids.shape[1]
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens, do_sample=False,
        return_dict_in_generate=True, output_scores=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    if FORBID_THINKING_TOKENS:
        suppress_ids = []
        for tok in ["<think>", "</think>"]:
            ids = tokenizer.encode(tok, add_special_tokens=False)
            if len(ids) == 1:
                suppress_ids.append(int(ids[0]))
        if suppress_ids:
            gen_kwargs["suppress_tokens"] = sorted(set(suppress_ids))
    with torch.no_grad():
        out = hf_model.generate(**inputs, **gen_kwargs)
    seq = out.sequences[0]
    new_ids = seq[input_len:].detach().cpu().tolist()
    raw = tokenizer.decode(new_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    clean = clean_generated_answer(raw)
    thinking_generated = "<think" in raw.lower() or "</think" in raw.lower()

    token_records = []
    content_logprobs = []
    first_content = None
    for i, tid in enumerate(new_ids):
        if i >= len(out.scores):
            break
        logits = out.scores[i][0].float()
        log_probs = torch.log_softmax(logits, dim=-1)
        prob = float(torch.exp(log_probs[int(tid)]).detach().cpu())
        logprob = float(log_probs[int(tid)].detach().cpu())
        rec = {
            "index": i, "token_id": int(tid),
            "token_text": decode_token(tid), "prob": prob, "logprob": logprob,
            "is_content": is_content_token(tid),
        }
        token_records.append(rec)
        if rec["is_content"]:
            content_logprobs.append(logprob)
            if first_content is None:
                first_content = rec
    return {
        "generated_answer_raw": raw,
        "generated_answer_clean": clean,
        "generated_answer_raw_repr": repr(raw),
        "generated_answer_clean_repr": repr(clean),
        "new_token_ids": [int(x) for x in new_ids],
        "token_records": token_records,
        "first_content_token_id": first_content["token_id"] if first_content else None,
        "first_content_token_text": first_content["token_text"] if first_content else None,
        "first_content_token_prob": first_content["prob"] if first_content else None,
        "first_content_token_logprob": first_content["logprob"] if first_content else None,
        "sequence_content_logprob_mean": float(np.mean(content_logprobs)) if content_logprobs else None,
        "sequence_content_logprob_sum": float(np.sum(content_logprobs)) if content_logprobs else None,
        "answer_length_tokens": len(new_ids),
        "answer_length_content_tokens": len(content_logprobs),
        "thinking_generated": thinking_generated,
        "prompt_mode": get_adapter(DATASET_ID).prompt_template_id,
    }


def select_workspace_layers(frac_start, frac_end):
    selected = []
    denom = max(n_layers - 1, 1)
    for layer in lens.source_layers:
        frac = layer / denom
        if frac_start <= frac <= frac_end:
            selected.append(layer)
    if LAYER_STRIDE > 1:
        selected = selected[::LAYER_STRIDE]
    return selected


workspace_layers_all = select_workspace_layers(WORKSPACE_LAYER_FRAC_START, WORKSPACE_LAYER_FRAC_END)
workspace_layers_mid = select_workspace_layers(MID_WORKSPACE_FRAC_START, MID_WORKSPACE_FRAC_END)
workspace_layers_late = select_workspace_layers(LATE_WORKSPACE_FRAC_START, LATE_WORKSPACE_FRAC_END)
# late_late band — narrower late-workspace slice, focused on
# the layers with the largest wrong-vs-correct entropy gap (L30-34
# on 36-layer Qwen3, per the PopQA layer-level analysis).
workspace_layers_late_late = select_workspace_layers(LATE_LATE_WORKSPACE_FRAC_START, LATE_LATE_WORKSPACE_FRAC_END)
print(f"Workspace layers all:      {workspace_layers_all[0]}–{workspace_layers_all[-1]} ({len(workspace_layers_all)} layers)")
print(f"Workspace layers mid:      {workspace_layers_mid[0]}–{workspace_layers_mid[-1]} ({len(workspace_layers_mid)} layers)")
print(f"Workspace layers late:     {workspace_layers_late[0]}–{workspace_layers_late[-1]} ({len(workspace_layers_late)} layers)")
print(f"Workspace layers late_late: {workspace_layers_late_late[0]}–{workspace_layers_late_late[-1]} ({len(workspace_layers_late_late)} layers)")


def rank_of_token(logits, token_id):
    if token_id is None:
        return None
    return int((logits > logits[int(token_id)]).sum().item()) + 1


def topk_features_from_logits(logits, k=FEATURE_TOP_K):
    top = torch.topk(logits, k=min(k, logits.numel()))
    top_logits = top.values.float()
    top_ids = top.indices.detach().cpu().tolist()
    p = torch.softmax(top_logits, dim=-1)
    entropy = float(-(p * torch.log(p + 1e-12)).sum().detach().cpu())
    effective = float(math.exp(entropy))
    top1_dom = float(p[0].detach().cpu())
    margin = float((top_logits[0] - top_logits[1]).detach().cpu()) if len(top_logits) > 1 else None
    return {
        "top_ids": [int(x) for x in top_ids],
        "top_tokens": [decode_token(x) for x in top_ids[:TOP_TOKENS_TO_SAVE]],
        "entropy_topk": entropy,
        "effective_concepts_topk": effective,
        "top1_dominance_topk": top1_dom,
        "top1_margin": margin,
    }


def jaccard(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def aggregate_band(layer_records, layers, prefix):
    subset = [r for r in layer_records if r["layer"] in set(layers)]
    out = {}
    if not subset:
        for name in [
            "entropy_topk_mean", "effective_concepts_topk_mean",
            "top1_dominance_topk_mean", "top1_margin_mean",
            "distinct_top1_count", "top1_churn_rate",
            "top5_jaccard_adjacent_mean", "entropy_topk_slope",
            "answer_token_best_rank",
        ]:
            out[f"{prefix}_{name}"] = None
        for name in [
            f"entropy_topk_last{'_' + str(COMMITMENT_LOOKBACK_LAYERS) if COMMITMENT_LOOKBACK_LAYERS != 3 else '3'}_slope",
            f"top1_dominance_last{'_' + str(COMMITMENT_LOOKBACK_LAYERS) if COMMITMENT_LOOKBACK_LAYERS != 3 else '3'}_max_minus_mean",
        ]:
            out[f"{prefix}_{name}"] = None
        return out
    def vals(key):
        return [r[key] for r in subset if r.get(key) is not None]
    ent = vals("entropy_topk")
    eff = vals("effective_concepts_topk")
    dom = vals("top1_dominance_topk")
    margin = vals("top1_margin")
    out[f"{prefix}_entropy_topk_mean"] = float(np.mean(ent)) if ent else None
    out[f"{prefix}_entropy_topk_std"] = float(np.std(ent)) if ent else None
    out[f"{prefix}_entropy_topk_min"] = float(np.min(ent)) if ent else None
    out[f"{prefix}_entropy_topk_max"] = float(np.max(ent)) if ent else None
    if len(ent) >= 2:
        xs = np.arange(len(ent))
        out[f"{prefix}_entropy_topk_slope"] = float(np.polyfit(xs, ent, 1)[0])
    else:
        out[f"{prefix}_entropy_topk_slope"] = None
    n_lookback = max(int(COMMITMENT_LOOKBACK_LAYERS), 1)
    late_subset = subset[-n_lookback:] if len(subset) >= n_lookback else subset
    ent_late = [r["entropy_topk"] for r in late_subset if r.get("entropy_topk") is not None]
    if len(ent_late) >= 2:
        xs_late = np.arange(len(ent_late))
        out[f"{prefix}_entropy_topk_last{'_' + str(n_lookback) if n_lookback != 3 else '3'}_slope"] = float(np.polyfit(xs_late, ent_late, 1)[0])
    else:
        out[f"{prefix}_entropy_topk_last{'_' + str(n_lookback) if n_lookback != 3 else '3'}_slope"] = None
    dom_late = [r["top1_dominance_topk"] for r in late_subset if r.get("top1_dominance_topk") is not None]
    if dom_late:
        out[f"{prefix}_top1_dominance_last{'_' + str(n_lookback) if n_lookback != 3 else '3'}_max_minus_mean"] = float(np.max(dom_late) - np.mean(dom_late))
    else:
        out[f"{prefix}_top1_dominance_last{'_' + str(n_lookback) if n_lookback != 3 else '3'}_max_minus_mean"] = None
    out[f"{prefix}_effective_concepts_topk_mean"] = float(np.mean(eff)) if eff else None
    out[f"{prefix}_top1_dominance_topk_mean"] = float(np.mean(dom)) if dom else None
    out[f"{prefix}_top1_margin_mean"] = float(np.mean(margin)) if margin else None
    top1s = [r["top1_token_id"] for r in subset]
    out[f"{prefix}_distinct_top1_count"] = int(len(set(top1s)))
    out[f"{prefix}_top1_churn_rate"] = (
        float(np.mean([top1s[i] != top1s[i-1] for i in range(1, len(top1s))]))
        if len(top1s) > 1 else 0.0
    )
    top5s = [r["top_ids"][:5] for r in subset]
    out[f"{prefix}_top5_jaccard_adjacent_mean"] = (
        float(np.mean([jaccard(top5s[i], top5s[i-1]) for i in range(1, len(top5s))]))
        if len(top5s) > 1 else None
    )
    ranks = [r["answer_token_rank"] for r in subset if r.get("answer_token_rank") is not None]
    out[f"{prefix}_answer_token_best_rank"] = int(min(ranks)) if ranks else None
    out[f"{prefix}_answer_token_mean_rank"] = float(np.mean(ranks)) if ranks else None
    return out




# Set to True to run the layer_l33_l34_features smoke test once at the
# end of cell 12. Slows down the run by ~1s but validates the helper's
# output schema on a synthetic example.
SMOKE_TEST_LAYER_FEATURES = False

def layer_l33_l34_features(layer_records):
    """Step B (v2.2): extract per-layer J-space features for L33 and L34,
    plus the L33-L34 entropy difference.

    Both L33 and L34 are inside workspace_layers_late_late (L30-L34 for
    Qwen3-4B). L33 is the layer with the strongest wrong-vs-correct signal
    per the v2.1 deep-dive; L34 is the final layer before readout. The
    diff captures the "commitment" flip in the last two layers.

    Returns a dict with 9 keys:
      - layer_l33_entropy_topk, layer_l33_top1_dominance_topk,
        layer_l33_top1_margin, layer_l33_answer_token_rank
      - layer_l34_entropy_topk, layer_l34_top1_dominance_topk,
        layer_l34_top1_margin, layer_l34_answer_token_rank
      - layer_l33_l34_entropy_diff
    """
    out = {}
    if LAYER_L33_TARGET_LAYER is None:
        for k in [
            "layer_l33_entropy_topk", "layer_l33_top1_dominance_topk",
            "layer_l33_top1_margin", "layer_l33_answer_token_rank",
            "layer_l34_entropy_topk", "layer_l34_top1_dominance_topk",
            "layer_l34_top1_margin", "layer_l34_answer_token_rank",
            "layer_l33_l34_entropy_diff",
        ]:
            out[k] = None
        return out

    def _find(records, target):
        for r in records:
            if r["layer"] == target:
                return r
        return None

    rec_l33 = _find(layer_records, LAYER_L33_TARGET_LAYER)
    rec_l34 = _find(layer_records, LAYER_L34_TARGET_LAYER)

    def _extract(rec, prefix):
        if rec is None:
            return {f"{prefix}_entropy_topk": None,
                    f"{prefix}_top1_dominance_topk": None,
                    f"{prefix}_top1_margin": None,
                    f"{prefix}_answer_token_rank": None}
        return {
            f"{prefix}_entropy_topk": float(rec.get("entropy_topk") or float("nan")),
            f"{prefix}_top1_dominance_topk": float(rec.get("top1_dominance_topk") or float("nan")),
            f"{prefix}_top1_margin": float(rec.get("top1_margin") or float("nan")),
            f"{prefix}_answer_token_rank": rec.get("answer_token_rank"),
        }

    out.update(_extract(rec_l33, "layer_l33"))
    out.update(_extract(rec_l34, "layer_l34"))

    e_l33 = out.get("layer_l33_entropy_topk")
    e_l34 = out.get("layer_l34_entropy_topk")
    if e_l33 is not None and e_l34 is not None:
        out["layer_l33_l34_entropy_diff"] = float(e_l33) - float(e_l34)
    else:
        out["layer_l33_l34_entropy_diff"] = None

    return out

def smoke_test_layer_l33_l34_features():
    """Validate layer_l33_l34_features() output schema.

    Builds a minimal layer_records list with one example at L33 and L34
    only, and checks that the helper emits the 9 expected columns with
    sensible values. Called once at the end of cell 12 if
    SMOKE_TEST_LAYER_FEATURES is True.
    """
    fake_records = [
        {"layer": LAYER_L33_TARGET_LAYER, "entropy_topk": 2.0, "top1_dominance_topk": 0.7,
         "top1_margin": 1.5, "answer_token_rank": 5},
        {"layer": LAYER_L34_TARGET_LAYER, "entropy_topk": 1.5, "top1_dominance_topk": 0.8,
         "top1_margin": 2.0, "answer_token_rank": 3},
    ]
    out = layer_l33_l34_features(fake_records)
    expected_keys = {
        "layer_l33_entropy_topk", "layer_l33_top1_dominance_topk",
        "layer_l33_top1_margin", "layer_l33_answer_token_rank",
        "layer_l34_entropy_topk", "layer_l34_top1_dominance_topk",
        "layer_l34_top1_margin", "layer_l34_answer_token_rank",
        "layer_l33_l34_entropy_diff",
    }
    actual_keys = set(out.keys())
    missing = expected_keys - actual_keys
    if missing:
        raise AssertionError(f"layer_l33_l34_features() is missing keys: {missing}")
    if out["layer_l33_l34_entropy_diff"] != 0.5:
        raise AssertionError(f"layer_l33_l34_entropy_diff should be 0.5, got {out['layer_l33_l34_entropy_diff']}")
    print("✅ layer_l33_l34_features() smoke test passed (9 keys, diff = 0.5).")
    return True



def extract_workspace_features(example_id, prompt, first_content_token_id):
    layers = workspace_layers_all
    lens_logits, _, input_ids_tensor = lens.apply(
        model, prompt, positions=[-1], layers=layers, max_seq_len=MAX_SEQ_LEN,
    )
    input_ids = input_ids_tensor[0].detach().cpu().tolist()
    readout_position = len(input_ids) - 1
    readout_prompt_token = decode_token(input_ids[readout_position])
    layer_records = []
    for layer in layers:
        logits = lens_logits[layer][0].float()
        f = topk_features_from_logits(logits, FEATURE_TOP_K)
        answer_rank = rank_of_token(logits, first_content_token_id)
        rec = {
            "example_id": example_id,
            "position_name": "pre_answer_final_prompt_token",
            "readout_position": readout_position,
            "readout_prompt_token": readout_prompt_token,
            "layer": int(layer),
            "top1_token_id": int(f["top_ids"][0]),
            "top1_token_text": f["top_tokens"][0] if f["top_tokens"] else None,
            "top_ids": f["top_ids"][:TOP_TOKENS_TO_SAVE],
            "top_tokens": f["top_tokens"],
            "entropy_topk": f["entropy_topk"],
            "effective_concepts_topk": f["effective_concepts_topk"],
            "top1_dominance_topk": f["top1_dominance_topk"],
            "top1_margin": f["top1_margin"],
            "answer_token_id": first_content_token_id,
            "answer_token_rank": answer_rank,
        }
        layer_records.append(rec)
    agg = {
        "example_id": example_id,
        "readout_position": readout_position,
        "readout_prompt_token": readout_prompt_token,
        "n_workspace_layers": len(layers),
    }
    agg.update(aggregate_band(layer_records, workspace_layers_all, "all"))
    agg.update(aggregate_band(layer_records, workspace_layers_mid, "mid"))
    agg.update(aggregate_band(layer_records, workspace_layers_late, "late"))

    agg.update(aggregate_band(layer_records, workspace_layers_late_late, "late_late"))
    agg["workspace_noise_score"] = (
        agg.get("late_late_entropy_topk_mean")
        if agg.get("late_late_entropy_topk_mean") is not None
        else agg.get("all_entropy_topk_mean")
    )

    agg.update(layer_l33_l34_features(layer_records))
    return layer_records, agg


def write_jsonl_record(rel_path, record, artifact_type="jsonl", description=""):
    artifact_run.append_jsonl(rel_path, record, artifact_type=artifact_type, description=description)


if SMOKE_TEST_LAYER_FEATURES:
    smoke_test_layer_l33_l34_features()
print("✅ Helpers ready.")
artifact_run.update_status(step="helpers_ready")


## 5. Load Dataset

Loads the dataset identified by `DATASET_ID` via the `jspace_adapters`
registry. Each adapter knows its own repo, config, split, alias-extraction
logic, prompt template, and per-dataset metadata.

The selected examples are stored as `JspaceExample` objects that carry
question, aliases, choices (for MC), correct choice label/text, and
dataset-specific metadata. They are written to `datasets/questions.jsonl`
and a manifest is written to `datasets/dataset_manifest.json`.

To switch datasets, change `DATASET_ID` in the configuration cell. The 7
available IDs are:
- `triviaqa_rc_nocontext` (the original baseline)
- `popqa`
- `nq_open`
- `truthfulqa_gen`
- `hotpotqa_distractor`
- `gsm8k`
- `commonsenseqa`


In [ ]:
from datasets import load_dataset
from huggingface_hub import list_repo_files

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Adapter-based dataset loading (registry refactor) ──────────────────────
# The legacy load_dataset_robust() + extract_aliases() pair is replaced by
# the jspace_adapters registry. Each adapter knows its own repo, config,
# split, and field-extraction logic. The main loop iterates over
# adapter.iter_examples() yielding JspaceExample objects.

print(f"Loading dataset {DATASET_ID} via registry...")
adapter = get_adapter(DATASET_ID)
adapter.load(n=N_EXAMPLES, seed=RANDOM_SEED)
selected = list(adapter.iter_examples())

# Build the manifest from the adapter (schema-versioned for meta-analysis).
dataset_manifest = adapter.manifest_dict(n=N_EXAMPLES, seed=RANDOM_SEED)
dataset_manifest.update({
    "n_kept_after_dedup": len(selected),
    "n_selected": len(selected),
    "examples_with_aliases": sum(1 for ex in selected if ex.aliases),
    "examples_with_choices": sum(1 for ex in selected if ex.choices),
    "random_seed": RANDOM_SEED,
    "run_mode": RUN_MODE,
    "dataset_role": DATASET_ROLE,
})
artifact_run.write_json("datasets/dataset_manifest.json", dataset_manifest, artifact_type="dataset_manifest", description="Dataset selection manifest.")

# Save each selected example (with metadata) to the questions.jsonl artefact.
for rec in selected:
    write_jsonl_record(
        "datasets/questions.jsonl",
        {
            "example_id": rec.example_id,
            "source_example_id": rec.source_example_id,
            "question": rec.question,
            "aliases": rec.aliases,
            "choices": rec.choices,
            "correct_choice_label": rec.correct_choice_label,
            "correct_choice_text": rec.correct_choice_text,
            "metadata": rec.metadata,
        },
        artifact_type="dataset",
        description="Selected examples (registry-format).",
    )

artifact_run.update_manifest_section("dataset", dataset_manifest)
artifact_run.update_status(step="dataset_loaded")

# Show a few examples.
for rec in selected[:3]:
    if rec.has_multiple_choice():
        print("-", rec.question, "=>", rec.correct_choice_label, rec.correct_choice_text)
    else:
        print("-", rec.question, "=>", rec.aliases[:3])


## 6. Mini Setup Check

Runs the already-validated boot-country probe quickly to catch obvious lens/model issues before the expensive main loop.


In [ ]:
mini_prompt = "Fact: The currency used in the country shaped like a boot is"
mini_layers = workspace_layers_late[-8:] if len(workspace_layers_late) >= 8 else workspace_layers_late
mini_lens_logits, _, _ = lens.apply(model, mini_prompt, positions=[-1], layers=mini_layers, max_seq_len=MAX_SEQ_LEN)
print("Mini setup check — top tokens at answer position in late layers:")
for layer in mini_layers:
    toks = [decode_token(t) for t in mini_lens_logits[layer][0].topk(5).indices.tolist()]
    print(f"L{layer:>3}: {toks}")
artifact_run.log_event("mini_setup_check_completed", {"layers": mini_layers})
artifact_run.update_status(step="mini_setup_check_completed")


## 7. Pilot Smoke Test

Runs a few examples end-to-end and prints concise diagnostics. This catches answer-formatting or label issues before the full loop.


In [ ]:
SMOKE_TEST_N = min(3, len(selected))
print(f"Running smoke test on {SMOKE_TEST_N} examples...")

# Helper: dispatch deterministic labeler based on the example shape.
def deterministic_dispatch(example, clean_answer):
    """Return the deterministic_label-style dict for one example + answer."""
    if example.has_multiple_choice():
        return deterministic_label_mc(clean_answer, example)
    if example.wants_prefix_match():
        return deterministic_label_with_prefix(clean_answer, example)
    return deterministic_label(clean_answer, example.aliases)


for rec in selected[:SMOKE_TEST_N]:
    prompt = build_prompt(rec, tokenizer=tokenizer)
    if rec is selected[0]:
        print("Prompt preview:", repr(prompt[-300:]))
    gen = generate_answer_with_confidence(prompt)
    label = label_correctness(
        gen["generated_answer_clean"], rec,
        question=rec.question,
        output_confidence=gen["first_content_token_prob"],
        example_id=rec.example_id,
        det=deterministic_dispatch(rec, gen["generated_answer_clean"]),
        record_judge=False,
    )
    print("=" * 80)
    print("Q:", rec.question)
    print("Generated raw:", repr(gen["generated_answer_raw"]))
    print("Generated clean:", repr(gen["generated_answer_clean"]))
    if rec.has_multiple_choice():
        print("Choices:", [(c["label"], c["text"]) for c in rec.choices])
        print("Correct:", rec.correct_choice_label, rec.correct_choice_text)
    else:
        print("Aliases:", rec.aliases[:5])
    print("Correct primary:", label["correct_primary"], "source:", label["label_source"], "quality:", label["label_quality"], "F1:", round(label["token_f1_max"], 3), "judge:", label.get("judge_verdict"), "regime:", label.get("label_regime", "strict"))
    print("First content token:", gen["first_content_token_text"], "prob:", gen["first_content_token_prob"])
    print("Thinking generated:", gen.get("thinking_generated"))

artifact_run.update_status(step="smoke_test_completed")


## 8. Main Experiment Loop

For each selected question:

1. build answer-only prompt,
2. generate deterministic answer,
3. compute output-confidence features,
4. label correctness by aliases,
5. extract J-space features at the pre-answer position,
6. save records incrementally.


In [ ]:
start_time = time.time()
print(f"Processing {len(selected)} examples...")

for idx, rec in enumerate(selected, start=1):
    example_id = rec.example_id
    source_example_id = rec.source_example_id
    question = rec.question
    aliases = rec.aliases
    prompt_id = f"prompt_{example_id}"
    # The adapter selects the prompt template (short_answer_v1, mc_letter_v1,
    # or numeric_v1) and includes choices for MC datasets.
    prompt = build_prompt(rec, tokenizer=tokenizer)

    record_prompt(
        prompt_id, "qa_baseline", prompt,
        prompt_type=get_adapter(DATASET_ID).prompt_template_id,
        spans={"pre_answer_position": {"token_positions": [-1]}},
        extra={
            "source_example_id": source_example_id,
            "question": question,
            "aliases": aliases[:20],
            "has_choices": rec.has_multiple_choice(),
            "correct_choice_label": rec.correct_choice_label,
        },
    )

    try:
        gen = generate_answer_with_confidence(prompt)
        # Dispatch deterministic labelling to the right helper.
        det = deterministic_dispatch(rec, gen["generated_answer_clean"])
        # label_correctness() accepts a precomputed det and a JspaceExample.
        label = label_correctness(
            gen["generated_answer_clean"], rec,
            question=question,
            output_confidence=gen["first_content_token_prob"],
            example_id=example_id,
            det=det,
        )
        layer_records, agg = extract_workspace_features(example_id, prompt, gen["first_content_token_id"])
    except Exception as e:
        err = {
            "example_id": example_id,
            "source_example_id": source_example_id,
            "question": question,
            "error_type": type(e).__name__,
            "error": str(e),
            "timestamp_utc": utc_now_iso(),
        }
        write_jsonl_record("logs/errors.jsonl", err, artifact_type="error", description="Per-example errors.")
        print(f"Error on example {idx}/{len(selected)} {example_id}: {type(e).__name__}: {e}")
        continue

    generation_record = {
        "example_id": example_id,
        "source_example_id": source_example_id,
        "prompt_id": prompt_id,
        "question": question,
        "aliases": aliases,
        "choices": rec.choices,
        "correct_choice_label": rec.correct_choice_label,
        "correct_choice_text": rec.correct_choice_text,
        **gen,
    }
    if not STORE_FULL_OUTPUTS:
        generation_record.pop("generated_answer_raw", None)
        generation_record.pop("generated_answer_clean", None)
    generation_records.append(generation_record)
    write_jsonl_record(
        "generations/generations.jsonl", generation_record,
        artifact_type="generation",
        description="Generated answers and output confidence records.",
    )

    for lr in layer_records:
        workspace_layer_records.append(lr)
        write_jsonl_record(
            "scans/workspace_layer_features.jsonl", lr,
            artifact_type="workspace_layer_features",
            description="Layer-level J-space feature rows.",
        )

    workspace_aggregate_records.append(agg)
    write_jsonl_record(
        "scans/workspace_aggregate_features.jsonl", agg,
        artifact_type="workspace_aggregate_features",
        description="Per-example aggregate J-space features.",
    )

    per_ex = {
        "example_id": example_id,
        "source_example_id": source_example_id,
        "source_dataset_index": rec.metadata.get("dataset_index"),
        "prompt_id": prompt_id,
        "question": question if STORE_FULL_PROMPTS else None,
        "generated_answer_clean": gen["generated_answer_clean"] if STORE_FULL_OUTPUTS else None,
        "aliases": aliases if STORE_FULL_PROMPTS else None,
        "choices": rec.choices,
        "correct_choice_label": rec.correct_choice_label,
        "correct_choice_text": rec.correct_choice_text,
        **{k: v for k, v in gen.items() if k not in {"token_records"}},
        **label,
        **agg,
    }
    per_ex["is_wrong"] = (label["correct_primary"] is False) if label.get("analysis_include_primary") else None
    per_ex["is_wrong_strict"] = not bool(label["correct_strict"])
    per_ex["is_wrong_expanded"] = not bool(label["correct_expanded"])
    per_ex["is_wrong_deterministic"] = not bool(label["correct_deterministic"])
    per_example_records.append(per_ex)
    write_jsonl_record(
        "metrics/per_example_metrics.jsonl", per_ex,
        artifact_type="per_example_metrics",
        description="Per-example correctness, confidence, and workspace features.",
    )

    if idx % SAVE_EVERY == 0 or idx == len(selected):
        elapsed = time.time() - start_time
        included = [r for r in per_example_records if r.get("analysis_include_primary")]
        acc = np.mean([bool(r["correct_primary"]) for r in included]) if included else float("nan")
        wrong = sum(1 for r in included if r.get("is_wrong"))
        excluded = len(per_example_records) - len(included)
        print(f"Processed {idx}/{len(selected)} | primary_accuracy={acc:.3f} | primary_wrong={wrong} | excluded={excluded} | elapsed={elapsed/60:.1f} min")
        artifact_run.update_status(
            step=f"processed_{idx}_examples",
            extra={
                "processed_examples": idx,
                "primary_accuracy_so_far": float(acc),
                "primary_wrong_so_far": int(wrong),
                "excluded_so_far": int(excluded),
            },
        )
        torch.cuda.empty_cache()

artifact_run.update_status(step="main_loop_completed", extra={"processed_examples": len(per_example_records)})
print("✅ Main loop completed.")


## 9. H.1 Analysis — Correct vs Incorrect and Predictive Models

Compares workspace features against output confidence and trains simple logistic-regression classifiers to predict wrong answers.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

per_df = pd.DataFrame(per_example_records)
agg_df = pd.DataFrame(workspace_aggregate_records)
layer_df = pd.DataFrame(workspace_layer_records)
label_judge_df = pd.DataFrame(label_judge_records)

artifact_run.write_dataframe("metrics/per_example_metrics", per_df, description="Per-example metrics table.")
artifact_run.write_dataframe("scans/workspace_aggregate_features", agg_df, description="Per-example aggregate J-space feature table.")
artifact_run.write_dataframe("scans/workspace_layer_features", layer_df, description="Layer-level J-space features.")
if len(label_judge_df):
    artifact_run.write_dataframe("metrics/label_judge_records", label_judge_df, description="LLM semantic label judge records.")

# Analysis label regimes.
def as_bool_series(s):
    return s.map(lambda x: True if x is True or str(x).lower() == "true" else (False if x is False or str(x).lower() == "false" else None))

per_df["correct_strict_bool"] = as_bool_series(per_df["correct_strict"])
per_df["correct_expanded_bool"] = as_bool_series(per_df["correct_expanded"])
per_df["correct_primary_bool"] = as_bool_series(per_df["correct_primary"])
per_df["analysis_include_primary_bool"] = per_df["analysis_include_primary"].map(lambda x: bool(x) and str(x).lower() != "false")
per_df["analysis_include_high_quality_bool"] = per_df["analysis_include_high_quality"].map(lambda x: bool(x) and str(x).lower() != "false")

label_regimes = {
    "strict": {"label_col": "correct_strict_bool", "mask": pd.Series([True] * len(per_df))},
    "expanded_deterministic": {"label_col": "correct_expanded_bool", "mask": pd.Series([True] * len(per_df))},
    "primary_with_llm_judge": {"label_col": "correct_primary_bool", "mask": per_df["analysis_include_primary_bool"]},
    "high_quality_primary": {"label_col": "correct_primary_bool", "mask": per_df["analysis_include_high_quality_bool"]},
}

primary_df = per_df[label_regimes["primary_with_llm_judge"]["mask"]].copy()
primary_df["is_wrong_primary"] = ~primary_df["correct_primary_bool"].astype(bool)

print(f"Examples processed: {len(per_df)}")
print(f"Primary included examples: {len(primary_df)}")
print(f"Primary correct: {int(primary_df['correct_primary_bool'].sum()) if len(primary_df) else 0}, Primary wrong: {int((~primary_df['correct_primary_bool'].astype(bool)).sum()) if len(primary_df) else 0}")
print(f"LLM judge calls: {len(label_judge_df)}")

label_summary_rows = []
for name, spec in label_regimes.items():
    df = per_df[spec["mask"]].copy()
    labels = as_bool_series(df[spec["label_col"]]).dropna()
    n = len(labels)
    label_summary_rows.append({
        "label_regime": name,
        "n": n,
        "n_correct": int(labels.sum()) if n else 0,
        "n_wrong": int((~labels.astype(bool)).sum()) if n else 0,
        "accuracy": float(labels.mean()) if n else None,
    })
label_summary_df = pd.DataFrame(label_summary_rows)
artifact_run.write_dataframe("metrics/label_regime_summary", label_summary_df, description="Summary of strict/expanded/primary label regimes.")
if display is not None:
    display(label_summary_df)
else:
    print(label_summary_df.to_string(index=False))

# Descriptive stats on primary labels.
key_features = [
    "first_content_token_prob",
    "sequence_content_logprob_mean",
    "workspace_noise_score",
    "late_entropy_topk_mean",
    "late_effective_concepts_topk_mean",
    "late_top1_dominance_topk_mean",
    "late_top1_margin_mean",
    "late_distinct_top1_count",
    "late_top1_churn_rate",
    "late_top5_jaccard_adjacent_mean",
]

def cohen_d(a, b):
    a = pd.Series(a).dropna().astype(float)
    b = pd.Series(b).dropna().astype(float)
    if len(a) < 2 or len(b) < 2:
        return None
    pooled = math.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return float((a.mean() - b.mean()) / pooled) if pooled > 0 else None

desc_rows = []
correct_df = primary_df[primary_df["correct_primary_bool"] == True]
wrong_df = primary_df[primary_df["correct_primary_bool"] == False]
for feat in key_features:
    if feat not in primary_df.columns:
        continue
    desc_rows.append({
        "feature": feat,
        "mean_correct": float(pd.to_numeric(correct_df[feat], errors="coerce").mean()),
        "mean_wrong": float(pd.to_numeric(wrong_df[feat], errors="coerce").mean()),
        "std_correct": float(pd.to_numeric(correct_df[feat], errors="coerce").std()),
        "std_wrong": float(pd.to_numeric(wrong_df[feat], errors="coerce").std()),
        "cohen_d_wrong_minus_correct": cohen_d(pd.to_numeric(wrong_df[feat], errors="coerce"), pd.to_numeric(correct_df[feat], errors="coerce")),
    })
desc_df = pd.DataFrame(desc_rows)
artifact_run.write_dataframe("metrics/feature_descriptive_stats", desc_df, description="Primary-label correct-vs-wrong descriptive feature statistics.")
if display is not None:
    display(desc_df)
else:
    print(desc_df.to_string(index=False))

# Predictive models by label regime.
baseline_features = ["first_content_token_prob", "sequence_content_logprob_mean", "answer_length_content_tokens"]
workspace_noise_single = ["workspace_noise_score"]
workspace_compact = ["late_entropy_topk_mean", "late_top1_dominance_topk_mean", "late_top1_churn_rate"]
workspace_full = [
    "late_entropy_topk_mean",
    "late_entropy_topk_std",
    "late_entropy_topk_slope",
    "late_effective_concepts_topk_mean",
    "late_top1_dominance_topk_mean",
    "late_top1_margin_mean",
    "late_distinct_top1_count",
    "late_top1_churn_rate",
    "late_top5_jaccard_adjacent_mean",
]
feature_sets = {
    "baseline_output_confidence": baseline_features,
    "workspace_noise_single": workspace_noise_single,
    "workspace_compact": workspace_compact,
    "workspace_full": workspace_full,
    "combined_compact": baseline_features + workspace_compact,
    "combined_full": baseline_features + workspace_full,

    "workspace_late_late_single": ["late_late_entropy_topk_mean"],
    "workspace_late_late_full": [
        "late_late_entropy_topk_mean", "late_late_entropy_topk_std",
        "late_late_entropy_topk_slope",
    ],
    "commitment_dynamics_single": ["late_top1_dominance_topk_mean"],
    "commitment_dynamics_full": [
        "late_top1_dominance_topk_mean",
        "late_top1_dominance_last3_max_minus_mean",
        "late_top1_churn_rate",
    ],

    "layer_l33_single": [f"layer_l{LAYER_L33_TARGET_LAYER}_entropy_topk"],
    "layer_l33_full": [
        f"layer_l{LAYER_L33_TARGET_LAYER}_entropy_topk",
        f"layer_l{LAYER_L33_TARGET_LAYER}_top1_dominance_topk",
        f"layer_l{LAYER_L33_TARGET_LAYER}_top1_margin",
    ],

    "layer_l33_diff": ["layer_l33_l34_entropy_diff"],
    "layer_l33_diff_in_full": [
        "layer_l33_l34_entropy_diff",
        f"layer_l{LAYER_L33_TARGET_LAYER}_entropy_topk",
        f"layer_l{LAYER_L33_TARGET_LAYER}_top1_dominance_topk",
    ],
}

auc_rows = []
cv_pred_rows = []
roc_plot_data = {}
analysis_status_by_regime = {}

for regime_name, spec in label_regimes.items():
    df = per_df[spec["mask"]].copy()
    labels = as_bool_series(df[spec["label_col"]])
    valid_mask = labels.notna()
    df = df[valid_mask].copy()
    y_correct = labels[valid_mask].astype(bool).values
    y = (~y_correct).astype(int)  # predict wrong
    min_class_count = int(min(np.bincount(y))) if len(set(y)) == 2 else 0
    if len(set(y)) < 2 or min_class_count < MIN_CLASS_COUNT_FOR_CV:
        analysis_status_by_regime[regime_name] = f"insufficient_class_balance_min_class={min_class_count}"
        print(f"⚠️ {regime_name}: insufficient class balance for CV (min_class={min_class_count}).")
        continue
    n_splits = min(CV_FOLDS, min_class_count)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    for set_name, feats in feature_sets.items():

        feats_original = list(feats)
        feats = [f for f in feats if f in df.columns]
        if not feats and feats_original:
            raise ValueError(
                f"Feature set {set_name!r} references columns not in DataFrame: {feats_original}. "
                "Check the helper in cell 12 and the feature_sets dict in cell 22 for column-name typos."
            )
        X = df[feats].apply(pd.to_numeric, errors="coerce").values
        oof = np.zeros(len(df), dtype=float)

        per_fold_aucs = []
        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y)):
            clf = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
            clf.fit(X[train_idx], y[train_idx])
            oof[test_idx] = clf.predict_proba(X[test_idx])[:, 1]
            if len(set(y[test_idx])) == 2:
                per_fold_aucs.append(float(roc_auc_score(y[test_idx], oof[test_idx])))
            for local_i in test_idx:
                cv_pred_rows.append({"example_id": df.iloc[local_i]["example_id"], "label_regime": regime_name, "feature_set": set_name, "fold": fold, "y_true_is_wrong": int(y[local_i]), "p_wrong": float(oof[local_i])})
        per_fold_std = float(np.std(per_fold_aucs, ddof=1)) if len(per_fold_aucs) > 1 else float("nan")
        roc_auc = float(roc_auc_score(y, oof))
        pr_auc = float(average_precision_score(y, oof))
        auc_rows.append({"label_regime": regime_name, "feature_set": set_name, "features": feats, "roc_auc": roc_auc, "roc_auc_per_fold_std": per_fold_std, "n_folds_with_both_classes": len(per_fold_aucs), "pr_auc": pr_auc, "n_splits": n_splits, "n": len(df), "n_wrong": int(y.sum())})
        if regime_name == "primary_with_llm_judge" and set_name in {"baseline_output_confidence", "workspace_compact", "combined_compact", "combined_full"}:
            roc_plot_data[f"{regime_name}:{set_name}"] = (y, oof)
        print(f"{regime_name} / {set_name}: ROC AUC={roc_auc:.3f}, PR AUC={pr_auc:.3f}")
    analysis_status_by_regime[regime_name] = "completed"

auc_df = pd.DataFrame(auc_rows)
cv_pred_df = pd.DataFrame(cv_pred_rows)
artifact_run.write_dataframe("metrics/auc_scores", auc_df, description="Cross-validated AUC scores by feature set and label regime.")
artifact_run.write_dataframe("metrics/cv_predictions", cv_pred_df, description="Out-of-fold predictions by label regime and feature set.")

# Bootstrap CIs and label-shuffle sanity checks over out-of-fold predictions.
bootstrap_rows = []
permutation_rows = []
if len(cv_pred_df):
    rng = np.random.default_rng(RANDOM_SEED)
    for (regime_name, set_name), g in cv_pred_df.groupby(["label_regime", "feature_set"]):
        y_arr = g["y_true_is_wrong"].astype(int).to_numpy()
        score_arr = g["p_wrong"].astype(float).to_numpy()
        if len(set(y_arr)) < 2:
            continue
        real_auc = roc_auc_score(y_arr, score_arr)
        real_pr = average_precision_score(y_arr, score_arr)
        boot_auc = []
        boot_pr = []
        for _ in range(BOOTSTRAP_N):
            idxs = rng.integers(0, len(y_arr), len(y_arr))
            if len(set(y_arr[idxs])) < 2:
                continue
            boot_auc.append(roc_auc_score(y_arr[idxs], score_arr[idxs]))
            boot_pr.append(average_precision_score(y_arr[idxs], score_arr[idxs]))
        if boot_auc:
            bootstrap_rows.append({
                "label_regime": regime_name,
                "feature_set": set_name,
                "roc_auc": float(real_auc),
                "roc_auc_ci_low": float(np.quantile(boot_auc, 0.025)),
                "roc_auc_ci_high": float(np.quantile(boot_auc, 0.975)),
                "pr_auc": float(real_pr),
                "pr_auc_ci_low": float(np.quantile(boot_pr, 0.025)),
                "pr_auc_ci_high": float(np.quantile(boot_pr, 0.975)),
                "bootstrap_n_effective": len(boot_auc),
            })
        perm_auc = []
        for _ in range(PERMUTATION_N):
            y_perm = rng.permutation(y_arr)
            if len(set(y_perm)) < 2:
                continue
            perm_auc.append(roc_auc_score(y_perm, score_arr))
        if perm_auc:
            permutation_rows.append({
                "label_regime": regime_name,
                "feature_set": set_name,
                "real_roc_auc": float(real_auc),
                "perm_auc_mean": float(np.mean(perm_auc)),
                "perm_auc_ci_low": float(np.quantile(perm_auc, 0.025)),
                "perm_auc_ci_high": float(np.quantile(perm_auc, 0.975)),
                "p_perm_ge_real": float(np.mean(np.array(perm_auc) >= real_auc)),
                "permutation_n": len(perm_auc),
            })

bootstrap_df = pd.DataFrame(bootstrap_rows)
permutation_df = pd.DataFrame(permutation_rows)
artifact_run.write_dataframe("metrics/auc_bootstrap_ci", bootstrap_df, description="Bootstrap confidence intervals for out-of-fold AUC metrics.")
artifact_run.write_dataframe("metrics/auc_permutation_sanity", permutation_df, description="Label-shuffle sanity checks for out-of-fold AUC metrics.")

# Final coefficients on primary combined compact/full if possible.
coef_records = []
if analysis_status_by_regime.get("primary_with_llm_judge") == "completed":
    df = primary_df.copy()
    y = (~df["correct_primary_bool"].astype(bool)).astype(int).values
    for model_name_coef, feats in {"combined_compact": baseline_features + workspace_compact, "combined_full": baseline_features + workspace_full}.items():
        feats = [f for f in feats if f in df.columns]
        X = df[feats].apply(pd.to_numeric, errors="coerce")
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        clf = LogisticRegression(max_iter=1000, class_weight="balanced")
        X_imp = imputer.fit_transform(X)
        X_scaled = scaler.fit_transform(X_imp)
        clf.fit(X_scaled, y)
        for feat, coef in zip(feats, clf.coef_[0]):
            coef_records.append({"label_regime": "primary_with_llm_judge", "feature_set": model_name_coef, "feature": feat, "coefficient_for_wrong": float(coef)})
coef_df = pd.DataFrame(coef_records)
artifact_run.write_dataframe("metrics/logistic_regression_coefficients", coef_df, description="Final logistic regression coefficients.")

artifact_run.write_json("metrics/analysis_status.json", {"analysis_status_by_regime": analysis_status_by_regime, "n_examples": len(per_df), "n_primary_included": len(primary_df)}, artifact_type="metrics", description="Predictive analysis status.")
artifact_run.update_status(step="h1_analysis_completed")


## 10. H.3 Analysis — Confident Hallucinations

Filters the H.1 table for high-confidence answers and compares clean vs noisy workspace states.


In [ ]:
quad_df = primary_df.copy()
quad_df["first_content_token_prob_num"] = pd.to_numeric(quad_df["first_content_token_prob"], errors="coerce")
quad_df["workspace_noise_score_num"] = pd.to_numeric(quad_df["workspace_noise_score"], errors="coerce")
quad_df["is_wrong_primary"] = ~quad_df["correct_primary_bool"].astype(bool)
if THRESHOLD_MODE == "frozen_absolute":
    import json
    with open(PROTOCOL_FILE, "r") as f:
        protocol = json.load(f)
    conf_threshold = protocol["thresholds"]["confidence_threshold"]
    noise_threshold = protocol["thresholds"]["noise_threshold"]
    print(f"Using frozen thresholds: conf={conf_threshold:.4f}, noise={noise_threshold:.4f}")
else:
    conf_threshold = float(quad_df["first_content_token_prob_num"].quantile(HIGH_CONFIDENCE_QUANTILE)) if len(quad_df) else float("nan")
    noise_threshold = float(quad_df["workspace_noise_score_num"].quantile(NOISY_WORKSPACE_QUANTILE)) if len(quad_df) else float("nan")
    print(f"Using exploratory quantiles: conf={conf_threshold:.4f}, noise={noise_threshold:.4f}")
quad_df["high_confidence"] = quad_df["first_content_token_prob_num"] >= conf_threshold
quad_df["workspace_noisy"] = quad_df["workspace_noise_score_num"] >= noise_threshold
quad_df["quadrant"] = quad_df.apply(lambda r: ("high" if r["high_confidence"] else "low") + "_confidence__" + ("noisy" if r["workspace_noisy"] else "clean") + "_workspace", axis=1)

quad_rows = []
for q, g in quad_df.groupby("quadrant"):
    quad_rows.append({
        "quadrant": q,
        "n": len(g),
        "accuracy": float(g["correct_primary_bool"].astype(bool).mean()) if len(g) else None,
        "wrong_rate": float(g["is_wrong_primary"].mean()) if len(g) else None,
        "mean_confidence": float(g["first_content_token_prob_num"].mean()),
        "mean_workspace_noise": float(g["workspace_noise_score_num"].mean()),
    })
quadrant_df = pd.DataFrame(quad_rows).sort_values("quadrant")
artifact_run.write_dataframe("metrics/quadrant_analysis", quadrant_df, description="H.3 high/low confidence by clean/noisy workspace quadrant analysis using primary labels.")
artifact_run.write_json("metrics/h3_thresholds.json", {"threshold_mode": THRESHOLD_MODE, "confidence_quantile": HIGH_CONFIDENCE_QUANTILE, "confidence_threshold": conf_threshold, "noise_quantile": NOISY_WORKSPACE_QUANTILE, "noise_threshold": noise_threshold, "label_regime": "primary_with_llm_judge"}, artifact_type="metrics", description="H.3 quadrant thresholds.")

if display is not None:
    display(quadrant_df)
else:
    print(quadrant_df.to_string(index=False))

buckets = {
    "confident_wrong_noisy": quad_df[(quad_df["high_confidence"]) & (quad_df["workspace_noisy"]) & (quad_df["is_wrong_primary"])].sort_values(["first_content_token_prob_num", "workspace_noise_score_num"], ascending=False).head(SELECTED_EXAMPLES_PER_BUCKET),
    "confident_wrong_clean_blindspots": quad_df[(quad_df["high_confidence"]) & (~quad_df["workspace_noisy"]) & (quad_df["is_wrong_primary"])].sort_values(["first_content_token_prob_num"], ascending=False).head(SELECTED_EXAMPLES_PER_BUCKET),
    "confident_correct_clean": quad_df[(quad_df["high_confidence"]) & (~quad_df["workspace_noisy"]) & (~quad_df["is_wrong_primary"])].sort_values(["first_content_token_prob_num"], ascending=False).head(SELECTED_EXAMPLES_PER_BUCKET),
    "possible_label_noise": per_df[
        (per_df["analysis_include_primary"] == False)
        | (per_df.get("temporal_unstable", False) == True)
        | (per_df.get("fuzzy_suggested", False) == True)
        | ((per_df["correct_strict_bool"] == False) & (per_df["correct_primary_bool"] == True))
        | ((per_df["correct_primary_bool"] == False) & (pd.to_numeric(per_df["token_f1_max"], errors="coerce") >= 0.5))
    ].head(SELECTED_EXAMPLES_PER_BUCKET),
}

def jspace_trajectory_markdown(example_id):
    if "layer_df" not in globals() or layer_df is None or len(layer_df) == 0:
        return ""
    lr = layer_df[layer_df["example_id"] == example_id].sort_values("layer")
    if len(lr) == 0:
        return ""
    late = lr[lr["layer"].isin(workspace_layers_late)]
    if len(late) == 0:
        late = lr.tail(15)
    lines = []
    lines.append("Late workspace top-1 trajectory:")
    lines.append("```text")
    traj = [f"L{int(r.layer)}:{r.top1_token_text}" for _, r in late.iterrows()]
    # Wrap roughly every five layers for readability.
    for i in range(0, len(traj), 5):
        lines.append("  " + " | ".join(traj[i:i+5]))
    lines.append("```")
    try:
        noisy = late.sort_values("entropy_topk", ascending=False).iloc[0]
        clean = late.sort_values("entropy_topk", ascending=True).iloc[0]
        lines.append("Noisiest late layer top tokens:")
        lines.append("```text")
        lines.append(f"L{int(noisy.layer)} entropy={float(noisy.entropy_topk):.3f}: {noisy.top_tokens}")
        lines.append("```")
        lines.append("Cleanest late layer top tokens:")
        lines.append("```text")
        lines.append(f"L{int(clean.layer)} entropy={float(clean.entropy_topk):.3f}: {clean.top_tokens}")
        lines.append("```")
    except Exception:
        pass
    return "\n".join(lines) + "\n"

def example_markdown(df, title):
    lines = [f"# {title}\n"]
    if len(df) == 0:
        lines.append("No examples in this bucket.\n")
        return "\n".join(lines)
    for _, r in df.iterrows():
        lines.append(f"## {r['example_id']}\n")
        lines.append(f"- Correct primary: `{r.get('correct_primary')}`")
        lines.append(f"- Strict correct: `{r.get('correct_strict')}`")
        lines.append(f"- Expanded correct: `{r.get('correct_expanded')}`")
        lines.append(f"- Label source: `{r.get('label_source')}`")
        lines.append(f"- Label quality: `{r.get('label_quality')}`")
        lines.append(f"- Temporal unstable: `{r.get('temporal_unstable')}`")
        lines.append(f"- Exclusion reason: `{r.get('label_exclusion_reason')}`")
        lines.append(f"- Fuzzy suggested: `{r.get('fuzzy_suggested')}`")
        lines.append(f"- Judge verdict: `{r.get('judge_verdict')}` ({r.get('judge_confidence')})")
        lines.append(f"- First content token prob: `{r.get('first_content_token_prob')}`")
        lines.append(f"- Workspace noise score: `{r.get('workspace_noise_score')}`")
        lines.append(f"- Quadrant: `{r.get('quadrant', '')}`")
        if STORE_SELECTED_EXAMPLES_TEXT:
            lines.append("\nQuestion:")
            lines.append("```text")
            lines.append(str(r.get("question")))
            lines.append("```")
            lines.append("Prediction:")
            lines.append("```text")
            lines.append(str(r.get("generated_answer_clean")))
            lines.append("```")
            lines.append("Aliases:")
            lines.append("```text")
            lines.append(str(r.get("aliases")))
            lines.append("```")
            if r.get("judge_reason"):
                lines.append("Judge reason:")
                lines.append("```text")
                lines.append(str(r.get("judge_reason")))
                lines.append("```")
            traj_md = jspace_trajectory_markdown(r.get("example_id"))
            if traj_md:
                lines.append(traj_md)
        lines.append("")
    return "\n".join(lines)

for name, dfb in buckets.items():
    artifact_run.write_dataframe(f"examples/{name}", dfb, description=f"Selected examples: {name}")
    artifact_run.write_text(f"examples/{name}.md", example_markdown(dfb, name.replace("_", " ").title()), artifact_type="example_report", description=f"Markdown selected examples: {name}")

artifact_run.update_status(step="h3_analysis_completed")


## 11. Plots

Creates compact diagnostic plots. These are supplementary; machine-readable metrics are primary.


In [ ]:
plot_paths = []
try:
    # ROC curves
    if roc_plot_data:
        plt.figure(figsize=(6, 5))
        for name, (yy, scores) in roc_plot_data.items():
            fpr, tpr, _ = roc_curve(yy, scores)
            auc_val = roc_auc_score(yy, scores)
            plt.plot(fpr, tpr, label=f"{name} AUC={auc_val:.3f}")
        plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
        plt.xlabel("False positive rate")
        plt.ylabel("True positive rate")
        plt.title("Wrong-answer prediction ROC")
        plt.legend()
        plt.tight_layout()
        p = artifact_run.full_path("plots/roc_curves.png")
        plt.savefig(p, dpi=160)
        plt.close()
        artifact_run.register_artifact("plots/roc_curves.png", artifact_type="plot", fmt="png", description="ROC curves for wrong-answer prediction.")
        plot_paths.append(str(p))

    # Workspace noise vs correctness
    plt.figure(figsize=(6, 4))
    data_correct = quad_df[quad_df["correct_primary_bool"] == True]["workspace_noise_score_num"].dropna()
    data_wrong = quad_df[quad_df["correct_primary_bool"] == False]["workspace_noise_score_num"].dropna()
    plt.boxplot([data_correct, data_wrong], labels=["correct", "wrong"])
    plt.ylabel("Workspace noise score")
    plt.title("Workspace noise by correctness")
    plt.tight_layout()
    p = artifact_run.full_path("plots/entropy_vs_correctness.png")
    plt.savefig(p, dpi=160)
    plt.close()
    artifact_run.register_artifact("plots/entropy_vs_correctness.png", artifact_type="plot", fmt="png", description="Workspace noise by correctness boxplot.")
    plot_paths.append(str(p))

    # Quadrant accuracy
    plt.figure(figsize=(8, 4))
    qplot = quadrant_df.copy()
    plt.bar(qplot["quadrant"], qplot["accuracy"])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Accuracy")
    plt.title("Accuracy by confidence/workspace quadrant")
    plt.tight_layout()
    p = artifact_run.full_path("plots/confidence_workspace_quadrants.png")
    plt.savefig(p, dpi=160)
    plt.close()
    artifact_run.register_artifact("plots/confidence_workspace_quadrants.png", artifact_type="plot", fmt="png", description="Accuracy by confidence/workspace quadrant.")
    plot_paths.append(str(p))

    print("Saved plots:")
    for p in plot_paths:
        print("-", p)
except Exception as e:
    print(f"Plotting failed: {type(e).__name__}: {e}")
    artifact_run.append_jsonl("logs/errors.jsonl", {"timestamp_utc": utc_now_iso(), "stage": "plots", "error_type": type(e).__name__, "error": str(e)}, artifact_type="error", description="Plotting errors.")

artifact_run.update_status(step="plots_completed")


## 12. Final Reports and Artefact Finalization

Writes `README.md`, `AGENT_BRIEF.md`, `FULL_REPORT.md`, updates global indexes, and finalizes the run.


In [ ]:
# Aggregate metrics.
auc_records = auc_df.to_dict(orient="records") if "auc_df" in globals() and len(auc_df) else []
aggregate_metrics = {
    "run_mode": RUN_MODE,
    "dataset_id": DATASET_ID,
    "dataset_role": DATASET_ROLE,
    "n_examples_requested": N_EXAMPLES,
    "n_examples_processed": len(per_df),
    "n_primary_included": len(primary_df),
    "accuracy": float(primary_df["correct_primary_bool"].mean()) if len(primary_df) else None,
    "wrong_rate": float((~primary_df["correct_primary_bool"].astype(bool)).mean()) if len(primary_df) else None,
    "n_correct": int(primary_df["correct_primary_bool"].sum()) if len(primary_df) else 0,
    "n_wrong": int((~primary_df["correct_primary_bool"].astype(bool)).sum()) if len(primary_df) else 0,
    "n_llm_judge_calls": len(label_judge_df) if "label_judge_df" in globals() else 0,
    "n_temporal_unstable": int(per_df.get("temporal_unstable", pd.Series(False, index=per_df.index)).astype(bool).sum()) if len(per_df) else 0,
    "n_primary_excluded": int((~per_df.get("analysis_include_primary", pd.Series(True, index=per_df.index)).astype(bool)).sum()) if len(per_df) else 0,
    "analysis_status_by_regime": analysis_status_by_regime,
    "auc_scores": auc_records,
    "confidence_threshold": conf_threshold,
    "noise_threshold": noise_threshold,
}
if auc_records:
    primary_auc = {r["feature_set"]: r["roc_auc"] for r in auc_records if r.get("label_regime") == "primary_with_llm_judge"}
    aggregate_metrics["delta_auc_combined_compact_minus_baseline"] = primary_auc.get("combined_compact", float("nan")) - primary_auc.get("baseline_output_confidence", float("nan"))
    aggregate_metrics["delta_auc_combined_full_minus_baseline"] = primary_auc.get("combined_full", float("nan")) - primary_auc.get("baseline_output_confidence", float("nan"))
artifact_run.write_json("metrics/aggregate_metrics.json", aggregate_metrics, artifact_type="metrics", description="Aggregate experiment metrics.")
artifact_run.write_json(
    "thresholds.json",
    {
        "high_confidence_quantile": HIGH_CONFIDENCE_QUANTILE,
        "noisy_workspace_quantile": NOISY_WORKSPACE_QUANTILE,
        "feature_top_k": FEATURE_TOP_K,
        "workspace_layer_frac_start": WORKSPACE_LAYER_FRAC_START,
        "workspace_layer_frac_end": WORKSPACE_LAYER_FRAC_END,
        "workspace_layers_all": workspace_layers_all,
        "workspace_layers_mid": workspace_layers_mid,
        "workspace_layers_late": workspace_layers_late,
        "label_regime_primary": "primary_with_llm_judge",
    },
    artifact_type="config",
    description="Thresholds and layer-band settings.",
)


def _ci_of(regime_name, feature_set_name):
    row = bootstrap_df[
        (bootstrap_df["label_regime"] == regime_name)
        & (bootstrap_df["feature_set"] == feature_set_name)
    ]
    if len(row) == 0:
        return (None, None, None)
    return (float(row["roc_auc"].iloc[0]),
            float(row["roc_auc_ci_low"].iloc[0]),
            float(row["roc_auc_ci_high"].iloc[0]))

base_auc, base_ci_lo, base_ci_hi = _ci_of("primary_with_llm_judge", "baseline_output_confidence")
cc_auc, cc_ci_lo, cc_ci_hi = _ci_of("primary_with_llm_judge", "combined_compact")
wn_auc, wn_ci_lo, wn_ci_hi = _ci_of("primary_with_llm_judge", "workspace_noise_single")

ci_strictly_above_baseline = (
    base_ci_hi is not None and wn_ci_lo is not None and wn_ci_lo > base_ci_hi
)
cc_ci_strictly_above_baseline = (
    base_ci_hi is not None and cc_ci_lo is not None and cc_ci_lo > base_ci_hi
)

# Interpret outcome using the primary label regime and compact combined model.
delta_auc = aggregate_metrics.get("delta_auc_combined_compact_minus_baseline")
primary_status = analysis_status_by_regime.get("primary_with_llm_judge", "not_run")
single_feature_subfinding = ""
if base_auc is not None and wn_auc is not None:
    wn_delta = wn_auc - base_auc
    if ci_strictly_above_baseline:
        single_feature_subfinding = (
            f" The single-feature `workspace_noise_score` AUC is {wn_auc:.3f} with CI "
            f"[{wn_ci_lo:.3f}, {wn_ci_hi:.3f}], entirely above the baseline CI's upper bound "
            f"({base_ci_hi:.3f}) — a cleaner statistical claim than the combined_compact delta."
        )
    else:
        single_feature_subfinding = (
            f" The single-feature `workspace_noise_score` AUC is {wn_auc:.3f} with CI "
            f"[{wn_ci_lo:.3f}, {wn_ci_hi:.3f}]; CI overlaps baseline upper bound "
            f"({base_ci_hi:.3f}), so the parsimonious claim is not statistically distinguishable either."
        )

if primary_status != "completed":
    conclusion_status = "inconclusive"
    conclusion = "Insufficient class balance for predictive CV analysis under the primary label regime. Inspect descriptive stats and selected examples."
elif cc_ci_strictly_above_baseline and delta_auc is not None and not math.isnan(delta_auc) and delta_auc > 0.03:
    conclusion_status = "supports_H_A_ci_distinguishable"
    conclusion = (
        "Combined workspace + confidence features improved AUC over output confidence with non-overlapping 95% bootstrap CIs under the primary label regime, supporting complementary J-space signal."
        + single_feature_subfinding
    )
# ── Operational Routing & Decile Analysis (Phase 0) ────────────────────────
import numpy as np

if len(primary_df) > 0 and not math.isnan(conf_threshold):
    pd.set_option('future.no_silent_downcasting', True)

    high_conf = quad_df[quad_df["high_confidence"]].copy()
    high_conf_clean = high_conf[~high_conf["workspace_noisy"]]
    high_conf_noisy = high_conf[high_conf["workspace_noisy"]]

    # 1. Odds Ratio & Gap
    acc_clean = high_conf_clean["correct_primary_bool"].mean()
    acc_noisy = high_conf_noisy["correct_primary_bool"].mean()
    gap = acc_clean - acc_noisy

    wrong_clean = (~high_conf_clean["correct_primary_bool"]).sum()
    correct_clean = high_conf_clean["correct_primary_bool"].sum()
    wrong_noisy = (~high_conf_noisy["correct_primary_bool"]).sum()
    correct_noisy = high_conf_noisy["correct_primary_bool"].sum()

    odds_noisy = wrong_noisy / correct_noisy if correct_noisy > 0 else float("inf")
    odds_clean = wrong_clean / correct_clean if correct_clean > 0 else float("inf")
    odds_ratio = odds_noisy / odds_clean if odds_clean > 0 else float("inf")

    or_df = pd.DataFrame([{
        "acc_clean": acc_clean, "acc_noisy": acc_noisy, "gap": gap,
        "odds_ratio": odds_ratio, "wrong_clean": wrong_clean,
        "correct_clean": correct_clean, "wrong_noisy": wrong_noisy,
        "correct_noisy": correct_noisy
    }])
    artifact_run.write_dataframe("metrics/quadrant_odds_ratio", or_df, description="Odds Ratio and Accuracy Gap for High-Conf Noisy vs Clean.")

    # 2. Deciles
    try:
        high_conf["noise_decile"] = pd.qcut(high_conf["workspace_noise_score_num"], 10, labels=False, duplicates='drop')
        decile_stats = high_conf.groupby("noise_decile").agg(
            n=("is_wrong_primary", "count"),
            wrong_rate=("is_wrong_primary", "mean"),
            mean_noise=("workspace_noise_score_num", "mean"),
            mean_conf=("first_content_token_prob_num", "mean")
        ).reset_index()
        artifact_run.write_dataframe("metrics/high_confidence_noise_deciles", decile_stats, description="Noise decile stratification of high-confidence predictions.")

        # We need spearman
        spearman = high_conf["workspace_noise_score_num"].corr(high_conf["is_wrong_primary"].astype(float), method='spearman')
    except Exception as e:
        print(f"Warning: Decile calculation failed (likely too few examples): {e}")
        spearman = float("nan")

    # 3. Routing at Budget
    total_high_conf = len(high_conf)
    total_target = high_conf["is_wrong_primary"].sum()

    def eval_routing(score_col, ascending=False):
        sorted_df = high_conf.sort_values(by=score_col, ascending=ascending)
        results = []
        for b in [0.05, 0.10, 0.20, 0.30]:
            n_esc = int(total_high_conf * b)
            esc_df = sorted_df.head(n_esc)
            n_caught = esc_df["is_wrong_primary"].sum()
            recall = n_caught / total_target if total_target > 0 else 0
            precision = n_caught / n_esc if n_esc > 0 else 0
            results.append({"budget": b, "n_esc": n_esc, "n_caught": n_caught, "recall": recall, "precision": precision})
        return pd.DataFrame(results)

    if total_high_conf > 0:
        conf_routing = eval_routing("first_content_token_prob_num", ascending=True)
        noise_routing = eval_routing("workspace_noise_score_num", ascending=False)

        c_mean, c_std = high_conf["first_content_token_prob_num"].mean(), high_conf["first_content_token_prob_num"].std()
        n_mean, n_std = high_conf["workspace_noise_score_num"].mean(), high_conf["workspace_noise_score_num"].std()

        if c_std > 0 and n_std > 0:
            high_conf["combined_linear_risk"] = ((high_conf["workspace_noise_score_num"] - n_mean) / n_std) - ((high_conf["first_content_token_prob_num"] - c_mean) / c_std)
            comb_routing = eval_routing("combined_linear_risk", ascending=False)
        else:
            comb_routing = pd.DataFrame()

        # Compile routing df
        routing_rows = []
        for b in [0.05, 0.10, 0.20, 0.30]:
            c_row = conf_routing[conf_routing["budget"] == b].iloc[0] if not conf_routing.empty else None
            n_row = noise_routing[noise_routing["budget"] == b].iloc[0] if not noise_routing.empty else None
            cb_row = comb_routing[comb_routing["budget"] == b].iloc[0] if not comb_routing.empty else None
            routing_rows.append({
                "budget": b,
                "recall_conf_only": float(c_row["recall"]) if c_row is not None else float("nan"),
                "precision_conf_only": float(c_row["precision"]) if c_row is not None else float("nan"),
                "recall_noise_only": float(n_row["recall"]) if n_row is not None else float("nan"),
                "precision_noise_only": float(n_row["precision"]) if n_row is not None else float("nan"),
                "recall_combined": float(cb_row["recall"]) if cb_row is not None else float("nan"),
                "precision_combined": float(cb_row["precision"]) if cb_row is not None else float("nan"),
            })
        routing_df = pd.DataFrame(routing_rows)
        artifact_run.write_dataframe("metrics/routing_at_budget", routing_df, description="Routing escalation recall/precision across budgets.")

        if display is not None:
            print("--- Routing at Budget ---")
            display(routing_df)

    # ── Conclusions ────────────────────────────────────────────────────────────
    # 1. Global AUC Conclusion
    if delta_auc is not None and not math.isnan(delta_auc) and delta_auc > 0.03 and (cc_ci_lo is not None and base_ci_hi is not None and cc_ci_lo > base_ci_hi):
        global_auc_conclusion = "Combined workspace + confidence features improved AUC over output confidence with non-overlapping 95% bootstrap CIs."
    elif delta_auc is not None and not math.isnan(delta_auc) and delta_auc > 0.03:
        global_auc_conclusion = "Point estimate exceeds the headline threshold (+0.03), but 95% bootstrap CIs overlap the baseline."
    else:
        global_auc_conclusion = "Combined features did not significantly outperform the output confidence baseline (delta AUC <= 0.03 or missing)."

    # 2. Stratification Conclusion
    odds_ratio = odds_ratio if 'odds_ratio' in locals() else float('nan')
    gap = gap if 'gap' in locals() else float('nan')
    spearman = spearman if 'spearman' in locals() else float('nan')

    strat_conclusion = (f"Odds Ratio: {odds_ratio:.2f}. " if not math.isnan(odds_ratio) else "Odds Ratio: N/A. ")
    strat_conclusion += (f"Accuracy Gap: {gap*100:.1f}pp. " if not math.isnan(gap) else "")
    strat_conclusion += (f"Noise vs Error Spearman: {spearman:.3f}." if not math.isnan(spearman) else "")

    # 3. Routing Conclusion
    try:
        r_10_c = float(routing_df[routing_df["budget"]==0.1]["recall_conf_only"].iloc[0])
        r_10_n = float(routing_df[routing_df["budget"]==0.1]["recall_noise_only"].iloc[0])
        routing_conclusion = f"At 10% budget, noise routing caught {r_10_n*100:.1f}% vs conf routing {r_10_c*100:.1f}%."
    except Exception:
        routing_conclusion = "Routing at 10% budget calculation failed or missing."

    aggregate_metrics["global_auc_conclusion"] = global_auc_conclusion
    aggregate_metrics["high_confidence_stratification_conclusion"] = strat_conclusion
    aggregate_metrics["routing_conclusion"] = routing_conclusion

    # Remove the legacy single conclusion field
    if "conclusion_status" in aggregate_metrics:
        del aggregate_metrics["conclusion_status"]
    if "conclusion" in aggregate_metrics:
        del aggregate_metrics["conclusion"]
if auc_records:
    primary_auc = {r["feature_set"]: r["roc_auc"] for r in auc_records if r.get("label_regime") == "primary_with_llm_judge"}
    aggregate_metrics["delta_auc_combined_compact_minus_baseline"] = primary_auc.get("combined_compact", float("nan")) - primary_auc.get("baseline_output_confidence", float("nan"))
    aggregate_metrics["delta_auc_combined_full_minus_baseline"] = primary_auc.get("combined_full", float("nan")) - primary_auc.get("baseline_output_confidence", float("nan"))
artifact_run.write_json("metrics/aggregate_metrics.json", aggregate_metrics, artifact_type="metrics", description="Aggregate experiment metrics.")
artifact_run.write_json(
    "thresholds.json",
    {
        "high_confidence_quantile": HIGH_CONFIDENCE_QUANTILE,
        "noisy_workspace_quantile": NOISY_WORKSPACE_QUANTILE,
        "feature_top_k": FEATURE_TOP_K,
        "workspace_layer_frac_start": WORKSPACE_LAYER_FRAC_START,
        "workspace_layer_frac_end": WORKSPACE_LAYER_FRAC_END,
        "workspace_layers_all": workspace_layers_all,
        "workspace_layers_mid": workspace_layers_mid,
        "workspace_layers_late": workspace_layers_late,
        "label_regime_primary": "primary_with_llm_judge",
    },
    artifact_type="config",
    description="Thresholds and layer-band settings.",
)



def _ci_of(regime_name, feature_set_name):
    row = bootstrap_df[
        (bootstrap_df["label_regime"] == regime_name)
        & (bootstrap_df["feature_set"] == feature_set_name)
    ]
    if len(row) == 0:
        return (None, None, None)
    return (float(row["roc_auc"].iloc[0]),
            float(row["roc_auc_ci_low"].iloc[0]),
            float(row["roc_auc_ci_high"].iloc[0]))

base_auc, base_ci_lo, base_ci_hi = _ci_of("primary_with_llm_judge", "baseline_output_confidence")
cc_auc, cc_ci_lo, cc_ci_hi = _ci_of("primary_with_llm_judge", "combined_compact")
wn_auc, wn_ci_lo, wn_ci_hi = _ci_of("primary_with_llm_judge", "workspace_noise_single")

ci_strictly_above_baseline = (
    base_ci_hi is not None and wn_ci_lo is not None and wn_ci_lo > base_ci_hi
)
cc_ci_strictly_above_baseline = (
    base_ci_hi is not None and cc_ci_lo is not None and cc_ci_lo > base_ci_hi
)

# Interpret outcome using the primary label regime and compact combined model.
delta_auc = aggregate_metrics.get("delta_auc_combined_compact_minus_baseline")
primary_status = analysis_status_by_regime.get("primary_with_llm_judge", "not_run")
single_feature_subfinding = ""
if base_auc is not None and wn_auc is not None:
    wn_delta = wn_auc - base_auc
    if ci_strictly_above_baseline:
        single_feature_subfinding = (
            f" The single-feature `workspace_noise_score` AUC is {wn_auc:.3f} with CI "
            f"[{wn_ci_lo:.3f}, {wn_ci_hi:.3f}], entirely above the baseline CI's upper bound "
            f"({base_ci_hi:.3f}) — a cleaner statistical claim than the combined_compact delta."
        )
    else:
        single_feature_subfinding = (
            f" The single-feature `workspace_noise_score` AUC is {wn_auc:.3f} with CI "
            f"[{wn_ci_lo:.3f}, {wn_ci_hi:.3f}]; CI overlaps baseline upper bound "
            f"({base_ci_hi:.3f}), so the parsimonious claim is not statistically distinguishable either."
        )

if primary_status != "completed":
    conclusion_status = "inconclusive"
    conclusion = "Insufficient class balance for predictive CV analysis under the primary label regime. Inspect descriptive stats and selected examples."
elif cc_ci_strictly_above_baseline and delta_auc is not None and not math.isnan(delta_auc) and delta_auc > 0.03:
    conclusion_status = "supports_H_A_ci_distinguishable"
    conclusion = (
        "Combined workspace + confidence features improved AUC over output confidence with non-overlapping 95% bootstrap CIs under the primary label regime, supporting complementary J-space signal."
        + single_feature_subfinding
    )
if delta_auc is not None and not math.isnan(delta_auc) and delta_auc > 0.03:
    conclusion_status = "supports_H_A_ci_overlap"
    conclusion = (
        "Point estimate exceeds the headline threshold (+0.03), but 95% bootstrap CIs overlap the baseline under the primary label regime. Treat the combined_compact finding as suggestive, not confirmed."
        + single_feature_subfinding
    )
if delta_auc is not None and not math.isnan(delta_auc) and abs(delta_auc) <= 0.03:
    conclusion_status = "supports_H_B_or_redundant"
    conclusion = (
        "Workspace features did not materially improve over output confidence under the primary label regime; signals may be redundant for this model/dataset."
        + single_feature_subfinding
    )
else:
    conclusion_status = "supports_H_C_or_no_signal"
    conclusion = (
        "Workspace features did not improve prediction over output confidence under the primary label regime in this run."
        + single_feature_subfinding
    )
aggregate_metrics["conclusion_status"] = conclusion_status
aggregate_metrics["conclusion"] = conclusion
artifact_run.write_json("metrics/aggregate_metrics.json", aggregate_metrics, artifact_type="metrics", description="Aggregate experiment metrics.")


single_feature_lines = ["## Headline (single-feature, parsimonious)\n"]
if base_auc is not None and wn_auc is not None:
    wn_delta = wn_auc - base_auc
    single_feature_lines.append(f"- `workspace_noise_score` (1 feature, late-layer mean entropy): **AUC {wn_auc:.3f}**")
    single_feature_lines.append(f"  vs `baseline_output_confidence` (3 features): **AUC {base_auc:.3f}** → delta **{wn_delta:+.3f}**")
    single_feature_lines.append(f"- 95% bootstrap CI for single-feature: [{wn_ci_lo:.3f}, {wn_ci_hi:.3f}]")
    single_feature_lines.append(f"- Baseline CI upper bound: {base_ci_hi:.3f}")
    if ci_strictly_above_baseline:
        single_feature_lines.append("- **CI strictly above baseline:** True (statistically distinguishable at 95%)")
    else:
        single_feature_lines.append("- **CI strictly above baseline:** False (CI overlaps baseline)")
else:
    single_feature_lines.append("_(single-feature AUCs not available; insufficient class balance)_")
single_feature_block = "\n".join(single_feature_lines) + "\n"
artifact_run.write_text("brief_single_feature.md", single_feature_block, artifact_type="brief", description="Rec 2: single-feature (parsimonious) AUC callout for AGENT_BRIEF.md.")

summary_df = pd.DataFrame([
    {"item": "model", "value": MODEL_NAME},
    {"item": "run_mode", "value": RUN_MODE},
    {"item": "dataset_id", "value": DATASET_ID},
    {"item": "dataset_role", "value": DATASET_ROLE},
    {"item": "n_processed", "value": len(per_df)},
    {"item": "n_primary_included", "value": len(primary_df)},
    {"item": "accuracy_primary", "value": aggregate_metrics["accuracy"]},
    {"item": "wrong_rate_primary", "value": aggregate_metrics["wrong_rate"]},
    {"item": "llm_judge_calls", "value": aggregate_metrics.get("n_llm_judge_calls")},
    {"item": "analysis_status_primary", "value": analysis_status_by_regime.get("primary_with_llm_judge")},
    {"item": "conclusion_status", "value": conclusion_status},
    {"item": "conclusion", "value": conclusion},
])
summary_md = dataframe_to_markdown(summary_df)
label_summary_md = dataframe_to_markdown(label_summary_df) if "label_summary_df" in globals() else "No label regime table.\n"
auc_md = dataframe_to_markdown(auc_df) if "auc_df" in globals() else "No AUC table.\n"
bootstrap_md = dataframe_to_markdown(bootstrap_df) if "bootstrap_df" in globals() else "No bootstrap CI table.\n"
permutation_md = dataframe_to_markdown(permutation_df) if "permutation_df" in globals() else "No permutation sanity table.\n"
quad_md = dataframe_to_markdown(quadrant_df) if "quadrant_df" in globals() else "No quadrant table.\n"
desc_md = dataframe_to_markdown(desc_df) if "desc_df" in globals() else "No descriptive stats.\n"

# Pull dataset details from the manifest (written by cell 14).
_dataset_repo = dataset_manifest.get("repo", "unknown") if "dataset_manifest" in globals() else "unknown"
_dataset_config = dataset_manifest.get("config") or "default"
_dataset_split = dataset_manifest.get("split", "unknown")
_dataset_task_family = dataset_manifest.get("task_family", "unknown")
_dataset_prompt_template = dataset_manifest.get("prompt_template_id", "unknown")
_dataset_max_new_tokens = dataset_manifest.get("max_new_tokens", "unknown")
_dataset_expected_failure = dataset_manifest.get("expected_failure_mode", "unknown")

readme = f"""# J-Space Run: {EXPERIMENT_ID}

Run ID: `{artifact_run.run_id}`

## Purpose

Hallucination detection via J-space state. Implements H.1 and H.3 on the `{DATASET_ID}` dataset (task family: `{_dataset_task_family}`).

## Model and runtime

- Model: `{MODEL_NAME}`
- dtype: `{resolved_dtype}`
- Lens: `{lens_filename}`
- GPU: `{gpu_name}`

## Dataset

- Dataset ID: `{DATASET_ID}`
- Repo: `{_dataset_repo}`
- Config: `{_dataset_config}`
- Split: `{_dataset_split}`
- Prompt template: `{_dataset_prompt_template}`
- Max new tokens: `{_dataset_max_new_tokens}`
- Expected failure mode: `{_dataset_expected_failure}`

## Summary

{summary_md}

## Label regimes

{label_summary_md}

## Key artefacts

- `AGENT_BRIEF.md`
- `FULL_REPORT.md`
- `datasets/dataset_manifest.json`
- `datasets/questions.jsonl`
- `generations/generations.jsonl`
- `scans/workspace_aggregate_features.csv`
- `scans/workspace_layer_features.csv`
- `metrics/per_example_metrics.csv`
- `metrics/auc_scores.csv`
- `metrics/auc_bootstrap_ci.csv`
- `metrics/auc_permutation_sanity.csv`
- `metrics/quadrant_analysis.csv`
- `examples/confident_wrong_noisy.md`
- `examples/confident_wrong_clean_blindspots.md`

## Caveats

- Correctness labels are automatic alias-based labels and may be noisy.
- Full vocabulary logits and residual streams are not saved by default.
- Workspace noise is measured with top-{FEATURE_TOP_K} entropy over selected workspace layers.
- CSV readers should use `keep_default_na=False` if exact string values such as "None" matter; JSONL files are authoritative for text fields.
"""
artifact_run.write_text("README.md", readme, artifact_type="report", description="Run README.")

agent_brief = f"""# Agent Brief — Hallucination Detection via J-Space

## Run metadata

- Run ID: `{artifact_run.run_id}`
- Experiment: `{EXPERIMENT_ID}` — {EXPERIMENT_NAME}
- Dataset ID: `{DATASET_ID}` (role: `{DATASET_ROLE}`)
- Dataset repo: `{_dataset_repo}` (config `{_dataset_config}`, split `{_dataset_split}`)
- Model: `{MODEL_NAME}`
- dtype: `{resolved_dtype}`
- Lens: `{lens_filename}`
- GPU: `{gpu_name}`
- Run mode: `{RUN_MODE}`
- Examples processed: `{len(per_df)}`

## Main conclusion

**{conclusion_status}** — {conclusion}

## Summary table

{summary_md}

## Label regimes

{label_summary_md}

## AUC table

{auc_md}

## Bootstrap AUC intervals

{bootstrap_md}

## Permutation sanity checks

{permutation_md}

## H.3 quadrant analysis

{quad_md}

## Rec 2 (single-feature callout)

See `brief_single_feature.md` for the parsimonious `workspace_noise_score` AUC + CI + strict-above-baseline check. The most defensible single-feature claim is in there.

## Rec 3 (layer-specific features, v2.1+)

**v2.2 note:** As of v2.2, `workspace_noise_score` IS the `late_late` band entropy (L30-L34), so `workspace_noise_single` and `workspace_late_late_single` are now the same field. The wider `late` band (L27-L34) is preserved as `late_entropy_topk_mean` for backwards compatibility. To compare bands side-by-side, look at `late_entropy_topk_mean` vs `late_late_entropy_topk_mean` directly in `workspace_aggregate_features.csv`.

**Layer-specific ablations** (also in `auc_scores.csv`):
- `commitment_dynamics_*`: commitment-dynamic features (max-minus-mean of top-1 dominance in last 3 layers).
- `layer_l33_single` / `layer_l33_full`: L33 alone entropy / full L33 triplet (entropy + dominance + margin). Tests whether the signal is concentrated at a single layer.
- `layer_l33_diff` / `layer_l33_diff_in_full`: L33 - L34 entropy difference, with and without the underlying L33 features. Tests the "commitment flip" hypothesis in the last two layers.

## Key machine-readable files

- `manifest.json`
- `config.json`
- `model_lens.json`
- `datasets/dataset_manifest.json`
- `datasets/questions.jsonl`
- `prompts/prompts.jsonl`
- `generations/generations.jsonl`
- `scans/workspace_aggregate_features.csv`
- `scans/workspace_layer_features.csv`
- `metrics/per_example_metrics.csv`
- `metrics/auc_scores.csv`
- `metrics/auc_bootstrap_ci.csv`
- `metrics/auc_permutation_sanity.csv`
- `metrics/cv_predictions.csv`
- `metrics/quadrant_analysis.csv`
- `metrics/aggregate_metrics.json`

## Selected example reports

- `examples/confident_wrong_noisy.md` — cases where J-space noise may catch confident errors.
- `examples/confident_wrong_clean_blindspots.md` — confident wrong answers with clean workspaces.
- `examples/confident_correct_clean.md` — high-confidence correct clean examples.
- `examples/possible_label_noise.md` — likely automatic-label edge cases.

## Caveats

- This is not a lie detector; it tests correlations between workspace features and correctness.
- Correctness labels are automatic and should be manually audited for key examples.
- Workspace entropy may reflect answer-type ambiguity, formatting, or tokenization, not only epistemic uncertainty.
- Display filters are optional and do not affect metrics. No NSFW-specific filtering is implemented.
- Full logits/residuals are not saved by default.
"""
artifact_run.write_text("AGENT_BRIEF.md", agent_brief, artifact_type="agent_brief", description="Concise briefing for AI agents and reviewers.")

full_report = f"""# Full Report — {EXPERIMENT_NAME}

## Research question

Do J-space activation patterns at answer time predict factual-answer correctness beyond output confidence?

## Configuration

```json
{json.dumps(initial_config, indent=2, ensure_ascii=False, default=str)}
```

## Dataset

- Dataset ID: `{DATASET_ID}`
- Repo: `{_dataset_repo}`
- Config: `{_dataset_config}`
- Split: `{_dataset_split}`
- Task family: `{_dataset_task_family}`
- Prompt template: `{_dataset_prompt_template}`

```json
{json.dumps(dataset_manifest, indent=2, ensure_ascii=False, default=str)}
```

## Aggregate summary

{summary_md}

## Label regimes

{label_summary_md}

## Descriptive statistics

{desc_md}

## Predictive AUC results

{auc_md}

## Bootstrap AUC intervals

{bootstrap_md}

## Permutation sanity checks

{permutation_md}

## H.3 quadrant analysis

{quad_md}

## Conclusion

**{conclusion_status}** — {conclusion}

## Artefact guide

See `AGENT_BRIEF.md` for the most useful machine-readable files and selected examples.
"""
artifact_run.write_text("FULL_REPORT.md", full_report, artifact_type="report", description="Full plaintext report.")

# Human-readable prompts report.
prompt_lines = ["# Prompt Records\n"]
for p in prompt_records.values():
    prompt_lines.append(f"## {p['prompt_id']}\n")
    prompt_lines.append(f"- Condition: `{p['condition_id']}`")
    prompt_lines.append(f"- Type: `{p['prompt_type']}`")
    prompt_lines.append(f"- Token count: {p['token_count']}\n")
    if STORE_FULL_PROMPTS and "prompt_text" in p:
        prompt_lines.append("```text")
        prompt_lines.append(p["prompt_text"])
        prompt_lines.append("```\n")
artifact_run.write_text("prompts/prompts.md", "\n".join(prompt_lines), artifact_type="prompt_report", description="Human-readable prompt records.")

artifact_run.update_manifest_section("conclusion", {"conclusion_status": conclusion_status, "conclusion": conclusion})
artifact_run.finalize(overall_result=conclusion_status, final_summary=conclusion)

print("=" * 80)
print("EXPERIMENT COMPLETE")
print("=" * 80)
print(f"Conclusion: {conclusion_status} — {conclusion}")
print(f"Artefacts saved to: {artifact_run.run_path}")
print("Key file for future review: AGENT_BRIEF.md")
